In [ ]:
import pandas as pd
import numpy as np

file_path = "AL_ML_AX_strain_data_251102.csv"
df = pd.read_csv(file_path)
df = df[(np.abs(df['tot_mag']) < 0.005) & (np.abs(df['ion1 tot']) > 0.0001) & (df['gamma point average splitting'] < 10) & (df['maximum splitting energy'] > 0.01)]
df = df.drop(columns = ['gamma point average splitting', 'ion1 tot','tot_mag'])

# 기존 이름과 새 이름을 매핑하는 딕셔너리 생성
rename_dict = {
    'maximum splitting energy': 'sse',
    'avg_bond_length': 'M_X_avg_bond_length',
    'max_bond_length': 'M_X_max_bond_length',
    'min_bond_length': 'M_X_min_bond_length',
    'std_bond_length': 'M_X_std_bond_length',
    'center_max_angle': 'XMX_max_angle',
    'center_min_angle': 'XMX_min_angle',
    'center_avg_angle': 'XMX_avg_angle',
    'center_std_angle': 'XMX_std_angle',
    'nonmag_max_angle': 'XXX_max_angle',
    'nonmag_min_angle': 'XXX_min_angle',
    'nonmag_std_angle': 'XXX_std_angle',
    'labelled_1st': 'inter_motif_dist_1',
    'labelled_2nd': 'inter_motif_dist_2',
    'labelled_3rd': 'inter_motif_dist_3',
    'global_1st': 'M_M_dist_1',
    'global_2nd': 'M_M_dist_2',
    'global_3rd': 'M_M_dist_3',
    'avg_long_axis': 'motif_long_axis',
    'avg_short_axis': 'motif_short_axis',
    'avg_axis_ratio': 'motif_axis_ratio',
    'motif0_nonmag_count': 'num_X_in_motif',
    'magnetic_atomic_number': 'Z_M',
    'magnetic_electronegativity': 'M_electronegativity',
    'nonmagnetic_atomic_number': 'Z_X',
    'nonmagnetic_electronegativity': 'X_electronegativity',
    'rotation_angle_deg': 'motif_alignment_angle',
    'avg_hull_volume': 'motif_volume',
    'avg_hull_area': 'motif_surface_area',
    'motif_unitcell_ratio': 'motif_unitcell_vol_ratio',
    'p_metric': 'min_match_rmsd',
    'p_metric_std': 'min_match_rmsd_std',
    'd_orb_e': 'M_d_nelec',
    'p_orb_e_non': 'X_p_nelec',
    'mag_charge': 'M_bader',
    'nonmag_charge': 'X_bader',
    'delta_Z': 'delta_Z',
    'abs_delta_Z': 'abs_delta_Z',
    'delta_q': 'delta_bader',
    'abs_delta_q': 'abs_delta_bader',
    'pd_ratio': 'p_d_elec_ratio',
    'ax_eq_gap': 'max_avg_len_diff',
    'bond_range': 'bond_len_range',
    'bond_cv': 'bond_len_cv',
    'center_angle_spread': 'XMX_angle_range',
    'nonmag_angle_spread': 'XXX_angle_range',
    'U_times_axeq': 'hubbard_u_x_len_diff',
    'delta_chi_times_axeq': 'abs_delta_chi_x_len_diff',
    'd_global_local_1st': 'mag_inter_dist_diff_1',
    'd_global_local_2nd': 'mag_inter_dist_diff_2',
    'd_global_local_3rd': 'mag_inter_dist_diff_3',
    'ion1 tot': 'M_magnet',
    'avg_s': 'motif_distortion',
    'avg_delta': 'offcenter_ratio'
}

# rename 메서드를 사용하여 컬럼명 변경
df_renamed = df.rename(columns=rename_dict)
df_renamed.to_csv('final_251102_data.csv', index=False)
df = df_renamed
df

In [ ]:
# 1. 상관계수 행렬 계산
corr_matrix = df.select_dtypes(include=np.number).corr(method='pearson')

# 2. 중복을 피하기 위해 상삼각 행렬 형태로 변환
#    (A-B 상관관계와 B-A 상관관계는 같으므로 하나만 남김)
sol = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                  .stack()
                  .sort_values(ascending=False))

# 3. 상관계수의 절대값이 0.8 이상인 쌍들만 출력
high_corr_pairs = sol[abs(sol) > 0.9]

#print("상관계수 TOP 10 쌍:")
#print(high_corr_pairs.head(10))

print("\n상관계수 절대값 0.8 이상인 모든 쌍:")
high_corr_pairs.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df['sse'],df['motif_distortion'])

In [ ]:
df.columns

In [ ]:
df = df[['filename', 'sse', 'min_match_rmsd', 'motif_unitcell_vol_ratio', 'p_d_elec_ratio',
  'M_M_dist_1', 'inter_motif_dist_1', 'min_match_rmsd_std', 'proxy_M_magnet', 'offcenter_ratio',
  'XMX_std_angle', 'XMX_avg_angle', 'delta_chi', 'XXX_std_angle', 'hungarian_rotation_angle_deg',
  'M_d_nelec', 'X_p_nelec']]

In [ ]:
# 1. 상관계수 행렬 계산
corr_matrix = df.select_dtypes(include=np.number).corr(method='pearson')

# 2. 중복을 피하기 위해 상삼각 행렬 형태로 변환
#    (A-B 상관관계와 B-A 상관관계는 같으므로 하나만 남김)
sol = (corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                  .stack()
                  .sort_values(ascending=False))

# 3. 상관계수의 절대값이 0.8 이상인 쌍들만 출력
high_corr_pairs = sol[abs(sol) > 0.8]

#print("상관계수 TOP 10 쌍:")
#print(high_corr_pairs.head(10))

print("\n상관계수 절대값 0.8 이상인 모든 쌍:")
high_corr_pairs.head(20)

In [ ]:
import os, re
import pandas as pd
out = df
# 제거할 strain 접두어 목록: 필요하면 추가 가능
SUFFIX_PREFIXES = ('st', 'x', 'y', 'z', 'a', 'b', 'c', 'strain', 'eps', 'ea', 'eb', 'ec', 'scale')

def extract_seed_id_from_filename_fixed(s: str) -> str:
    s = str(s)
    stem = os.path.splitext(os.path.basename(s))[0]   # 확장자/경로 제거 -> 파일명 본체
    toks = stem.split('_')
    # 뒤에서부터 strain 토큰을 제거 (예: _st950, _x975 등)
    while len(toks) > 1:
        last = toks[-1]
        # 접두어가 목록에 있고 뒤가 숫자/부호/소수점이면 strain 토큰으로 간주
        if re.match(r'^(?:' + '|'.join(SUFFIX_PREFIXES) + r')[\+\-]?\d*(?:\.\d+)?$', last, flags=re.I):
            toks.pop()
        else:
            break
    # 주의: 마지막 토큰이 '3'처럼 **숫자만**인 경우(폴리모프 ID)는 **유지**한다.
    return '_'.join(toks)

# 적용
groups = out['filename'].astype(str).map(extract_seed_id_from_filename_fixed)

print("수정 후 그룹(씨드) 유니크 개수:", groups.nunique())
print(pd.DataFrame({'filename': out['filename'].head(10).astype(str),
                    'seed_id': groups.head(10)}))


In [ ]:
np.random.randint(100)

In [ ]:
# ============================================================
# StratifiedGroupKFold OOF + Optuna (Bayesian Optimization) + Final model + SHAP
# (XGBoost 2.x; Stratified by log1p(SSE))
# ============================================================
import os, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from xgboost import XGBRegressor
# [수정] GroupKFold와 함께 StratifiedGroupKFold를 import합니다.
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.metrics import mean_squared_error, r2_score
import shap
# [추가] Optuna를 import 합니다.
import optuna
from optuna.samplers import TPESampler

print("[INFO] xgboost version =", xgb.__version__)

# ---------------------------
# 0) 설정
# ---------------------------
USE_QUICK_FEATURES = True      # (원본 코드 기능 유지)
TRANSFORM_KIND     = 'log1p'   # 'raw' | 'log1p' | 'asinh'
USE_SMEARING       = True      # 스미어링 사용을 권장하여 True로 변경
N_SPLITS           = 10        # GroupKFold 폴드 수 (<= 유니크 seed 수)
INNER_SPLITS       = 5         # 내부 GroupKFold 분할 수 (early stopping 용)
SEED_BASE          = 102
# Hyperparam search
N_HP_TRIALS        = 200       # 랜덤 서치 trial 수
N_HP_RUNS          = 3         # 각 trial에 대해 반복 GroupKFold 횟수(폴드 다양화)
N_OOF_RUNS_BEST    = 5         # 베스트 파라미터로 최종 OOF 생성 시 반복 횟수 (안정성을 위해 5회로 늘림)
SAVE_DIR           = "20251101_stratified_groupkfold_BO"
os.makedirs(SAVE_DIR, exist_ok=True)

# ---------------------------
# 2) 데이터 로드 & 피처 구성
# ---------------------------
df = out.copy()  # 사용자의 DataFrame (메모리에 존재한다고 가정)

target = 'sse'
assert 'filename' in df.columns and target in df.columns, "df에 filename/target 컬럼이 필요합니다."

# 숫자 피처만 사용 (filename/target 제외)
num_cols = df.drop(columns=['filename', target], errors='ignore') \
             .select_dtypes(include=[np.number]).columns.tolist()
X_all = df[num_cols].values
y_all = df[target].values

# ---------------------------
# 3) seed(그룹) 라벨 추출
# ---------------------------
SUFFIX_PREFIXES = ('st','x','y','z','a','b','c','strain','eps','ea','eb','ec','scale')
def extract_seed_id_from_filename_fixed(s: str) -> str:
    stem = os.path.splitext(os.path.basename(str(s)))[0]
    toks = stem.split('_')
    while len(toks) > 1:
        last = toks[-1]
        if re.match(r'^(?:' + '|'.join(SUFFIX_PREFIXES) + r')[\+\-]?\d*(?:\.\d+)?$', last, flags=re.I):
            toks.pop()
        else:
            break
    return '_'.join(toks)

groups = df['filename'].astype(str).map(extract_seed_id_from_filename_fixed)
unique_groups = groups.unique()
n_groups = groups.nunique()
assert n_groups >= 2, "그룹(씨드) 수가 2 미만이면 GroupKFold 불가"
print(f"[INFO] unique seeds = {n_groups}, samples = {len(df)}, features = {len(num_cols)}")

# (선택) 씨드 과대표집 완화를 위한 샘플 가중치
seed_counts = groups.value_counts()
sample_weight_all = groups.map(1.0 / seed_counts).values

# =================================================================
# [추가] 중요한 데이터(SSE > 1)에 가중치 부여
# =================================================================
# SSE > 1인 데이터에 부여할 가중치 배율 (이 값을 튜닝할 수 있습니다)
high_sse_multiplier = 1

# y_all (df[target].values)을 사용하여 조건에 맞는 가중치를 생성합니다.
sse_weights = np.ones(len(df))
sse_weights[y_all > 1.0] = high_sse_multiplier

# 기존의 그룹 크기 보정 가중치와 SSE 중요도 가중치를 곱하여 최종 가중치를 만듭니다.
sample_weight_all = sample_weight_all * sse_weights

# 가중치가 잘 적용되었는지 확인 (선택 사항)
print(f"[INFO] Sample weights adjusted for high SSE values. Min: {sample_weight_all.min():.2f}, Max: {sample_weight_all.max():.2f}")
# =================================================================

# ---------------------------
# 4) 타깃 변환 / 역변환 / 스미어링 정의
# ---------------------------
def get_transform(kind):
    if kind == 'raw':
        f = lambda y: y
        finv = lambda z: z
        smearable = False
    elif kind == 'log1p':
        f = lambda y: np.log1p(np.maximum(y, 0))
        finv = lambda z: np.expm1(z)
        smearable = True
    elif kind == 'asinh':
        f = lambda y: np.arcsinh(y)
        finv = lambda z: np.sinh(z)
        smearable = False
    else:
        raise ValueError("TRANSFORM_KIND must be in {'raw','log1p','asinh'}")
    return f, finv, smearable

f_t, finv_t, smearable = get_transform(TRANSFORM_KIND)

# ---------------------------
# [추가] StratifiedGroupKFold를 위한 층(strata) 생성
# ---------------------------
# 모델이 학습하는 log1p 변환된 SSE 값의 그룹별 평균을 기준으로 층을 나눕니다.
group_log_sse_means = df.groupby(groups)[target].apply(lambda sse: f_t(sse).mean())
y_group_mean_log = groups.map(group_log_sse_means)

# 연속적인 값을 N_SPLITS개의 이산적인 구간으로 나눕니다 (층 생성).
y_stratify = pd.qcut(y_group_mean_log, q=N_SPLITS, labels=False, duplicates='drop')

# qcut으로 충분한 수의 bin이 생성되지 않을 경우 cut으로 재시도합니다.
if y_stratify.nunique() < 2:
    y_stratify = pd.cut(y_group_mean_log, bins=min(N_SPLITS, y_group_mean_log.nunique()), labels=False, duplicates='drop')
if y_stratify.nunique() < 2:
    print(f"[WARNING] Stratification failed (only {y_stratify.nunique()} bins). Performance may be unstable.")

print(f"[INFO] Stratification bins (on {TRANSFORM_KIND} scale) created with {y_stratify.nunique()} unique strata.")


# ---------------------------
# 5) 내부 Group-valid 분할 유틸 (train 그룹만 대상으로 1회 분할)
# ---------------------------
def make_inner_group_split(train_groups_vec, inner_splits=5):
    ug = np.array(pd.unique(train_groups_vec))
    n_s = min(max(2, inner_splits), len(ug))
    gkf = GroupKFold(n_splits=n_s)
    # 그룹 수가 분할 수보다 적으면 단일 폴드로 처리 (검증 없이)
    if len(ug) < n_s:
        return np.ones(len(train_groups_vec), dtype=bool), np.zeros(len(train_groups_vec), dtype=bool)
    
    tr_g_idx, va_g_idx = next(gkf.split(ug, groups=ug))  # 첫 split 사용
    tr_g = set(ug[tr_g_idx]); va_g = set(ug[va_g_idx])
    tr_mask = train_groups_vec.isin(tr_g).values
    va_mask = train_groups_vec.isin(va_g).values
    return tr_mask, va_mask

# ---------------------------
# 6) 타입 안전: 파라미터 캐스팅 & 모델 빌더 (2.x 전용)
# ---------------------------
INT_KEYS = {"max_depth", "min_child_weight", "n_estimators"}
FLOAT_KEYS = {"learning_rate", "subsample", "colsample_bytree", "reg_lambda", "reg_alpha"}

def cast_params(params: dict) -> dict:
    safe = {}
    for k, v in params.items():
        if k in INT_KEYS:
            safe[k] = int(v)
        elif k in FLOAT_KEYS:
            safe[k] = float(v)
        else:
            safe[k] = v
    return safe

def build_model(params: dict, n_estimators: int | None = None, random_state: int = 42) -> XGBRegressor:
    p = cast_params(params.copy())
    if n_estimators is not None:
        p['n_estimators'] = int(n_estimators)
    p.setdefault('n_estimators', 5000)
    p.setdefault('learning_rate', 0.03)
    p.setdefault('max_depth', 6)
    p.setdefault('min_child_weight', 3)
    p.setdefault('subsample', 0.8)
    p.setdefault('colsample_bytree', 0.8)
    p.setdefault('reg_lambda', 1.0)
    p.setdefault('reg_alpha', 0.0)
    random_state = np.random.randint(1000)
    # XGBoost 2.x: eval_metric / early_stopping_rounds는 생성자에 지정
    return XGBRegressor(
        **p,
        n_jobs=-1, # n_jobs=-1로 변경하여 병렬 처리 활용
        random_state=int(random_state),
        tree_method='hist',
        verbosity=0,
        eval_metric='rmse',
        early_stopping_rounds=100
    )

# ---------------------------
# 8) 한 trial의 점수 계산 (N_HP_RUNS 반복 StratifiedGroupKFold) - [수정됨]
# ---------------------------
def score_params(params, n_runs=N_HP_RUNS, seed_base=SEED_BASE):
    n_splits_eff = min(N_SPLITS, n_groups)
    if n_splits_eff < 2:
        raise ValueError("유니크 seed 수가 부족합니다. N_SPLITS를 줄이세요.")

    run_r2, run_mse = [], []
    all_best_iters, all_smears = [], []

    for run_idx in range(n_runs):
        # [수정] StratifiedGroupKFold를 사용합니다. shuffle과 random_state로 분할 다양성을 확보합니다.
        sgkf_outer = StratifiedGroupKFold(n_splits=n_splits_eff,
                                          shuffle=True,
                                          random_state=seed_base + 1000 * run_idx)

        oof_sum = np.zeros(len(df), dtype=float)
        oof_cnt = np.zeros(len(df), dtype=int)

        # [수정] split 호출이 간단해집니다. X, y_stratify, groups를 전달합니다.
        for fold_idx, (tr, te) in enumerate(
            sgkf_outer.split(X_all, y_stratify, groups=groups)
        ):
            # tr, te 인덱스는 바로 사용할 수 있습니다.
            X_tr, X_te = X_all[tr], X_all[te]
            y_tr, y_te = y_all[tr], y_all[te]
            sw_tr      = sample_weight_all[tr]
            g_tr       = groups.iloc[tr]

            # 내부 그룹-검증 분할(early stopping)
            tr_mask_in, va_mask_in = make_inner_group_split(g_tr, inner_splits=INNER_SPLITS)
            
            # eval_set과 그에 맞는 가중치를 준비합니다.
            eval_set = [(X_tr[va_mask_in], f_t(y_tr[va_mask_in]))] if np.any(va_mask_in) else None
            # [추가] XGBoost는 eval 가중치를 리스트 형태로 받습니다.
            eval_wts = [sw_tr[va_mask_in]] if np.any(va_mask_in) else None

            model = build_model(params, n_estimators=5000, random_state=42)
            model.fit(
                X_tr[tr_mask_in], f_t(y_tr[tr_mask_in]),
                # 훈련 데이터에 대한 가중치
                sample_weight=sw_tr[tr_mask_in],
                # 검증 데이터 셋
                eval_set=eval_set,
                # [추가] 검증 데이터에 대한 가중치
                sample_weight_eval_set=eval_wts,
                verbose=False
            )

            # 조기 종료가 작동하지 않은 경우, n_estimators를 best_iteration으로 사용
            bi = int(getattr(model, 'best_iteration', model.n_estimators)) + 1
            all_best_iters.append(bi)
            
            c = 1.0
            if TRANSFORM_KIND == 'log1p' and USE_SMEARING and np.any(va_mask_in):
                z_val = f_t(y_tr[va_mask_in])
                z_hat = model.predict(X_tr[va_mask_in], iteration_range=(0, bi))
                resid = z_val - z_hat
                c = float(np.mean(np.exp(resid)))
                all_smears.append(c)

            z_pred_te = model.predict(X_te, iteration_range=(0, bi))
            y_pred_te = finv_t(z_pred_te)
            if TRANSFORM_KIND == 'log1p' and USE_SMEARING:
                y_pred_te = c * (y_pred_te + 1.0) - 1.0

            oof_sum[te] += y_pred_te
            oof_cnt[te] += 1

        valid = oof_cnt > 0
        oof_pred_run = np.zeros_like(oof_sum)
        oof_pred_run[valid] = oof_sum[valid] / oof_cnt[valid]

        run_r2.append(r2_score(y_all[valid], oof_pred_run[valid]))
        run_mse.append(mean_squared_error(y_all[valid], oof_pred_run[valid]))

    summary = {
        "r2_mean": np.mean(run_r2), "r2_std": np.std(run_r2),
        "mse_mean": np.mean(run_mse), "mse_std": np.std(run_mse),
        "best_iter_median": int(np.median(all_best_iters)) if all_best_iters else 800,
        "smearing_mean": np.mean(all_smears) if all_smears else 1.0
    }
    return summary

# ---------------------------
# 9) 하이퍼파라미터 탐색 (베이지안 최적화: Optuna/TPE)
# ---------------------------
optuna_sampler = TPESampler(seed=SEED_BASE)

def objective(trial: optuna.trial.Trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        "reg_alpha":  trial.suggest_float("reg_alpha",  1e-3, 1.0,  log=True),
    }
    score = score_params(params, n_runs=N_HP_RUNS, seed_base=SEED_BASE + 100000*trial.number)
    for k, v in score.items():
        trial.set_user_attr(k, v)
    return score["r2_mean"]

print(f"[HP-SEARCH/BO] trials={N_HP_TRIALS}, runs/trial={N_HP_RUNS}, outer_folds={min(N_SPLITS, n_groups)}")
study = optuna.create_study(direction="maximize", sampler=optuna_sampler)
study.optimize(objective, n_trials=N_HP_TRIALS, show_progress_bar=True, n_jobs=1) # n_jobs=1로 해야 재현성 보장

def trials_to_hp_df(study: optuna.Study) -> pd.DataFrame:
    rows = [
        {**tr.params, "r2_mean": float(tr.value), **tr.user_attrs}
        for tr in study.trials if tr.state == optuna.trial.TrialState.COMPLETE
    ]
    if not rows:
        raise RuntimeError("Optuna study has no completed trials.")
    df_ = pd.DataFrame(rows)
    return df_.sort_values(["r2_mean", "mse_mean"], ascending=[False, True]).reset_index(drop=True)

hp_df = trials_to_hp_df(study)
hp_path = os.path.join(SAVE_DIR, "hp_search_results.csv")
hp_df.to_csv(hp_path, index=False)
print(f"[SAVE] {hp_path}")

best_row = hp_df.iloc[0].to_dict()
best_params = {k: v for k, v in best_row.items() if k in FLOAT_KEYS or k in INT_KEYS}
best_iter_from_search = int(best_row.get('best_iter_median', 800))
c_cv_from_search      = float(best_row.get('smearing_mean', 1.0))

print("\n[HP-SEARCH/BO] Best params (type-safe):")
print(json.dumps(cast_params(best_params), indent=2))
print(f"[HP-SEARCH/BO] best_iter_median={best_iter_from_search}, smearing_mean={c_cv_from_search:.4f}")

# ---------------------------
# 10) 베스트 파라미터로 최종 OOF 생성(여러 run 평균) - [수정됨]
# ---------------------------
def make_oof_with_params(params, n_runs=N_OOF_RUNS_BEST):
    n_splits_eff = min(N_SPLITS, n_groups)
    oof_sum = np.zeros(len(df), dtype=float)
    oof_cnt = np.zeros(len(df), dtype=int)
    fold_mse, fold_r2, best_iters, smear_factors = [], [], [], []

    for run_idx in range(n_runs):
        # [수정] StratifiedGroupKFold 사용
        sgkf_outer = StratifiedGroupKFold(n_splits=n_splits_eff,
                                          shuffle=True,
                                          random_state=SEED_BASE + 5555 * run_idx)

        # [수정] split 호출 변경
        for fold_idx, (tr, te) in enumerate(
            sgkf_outer.split(X_all, y_stratify, groups=groups)
        ):
            X_tr, X_te = X_all[tr], X_all[te]
            y_tr, y_te = y_all[tr], y_all[te]
            sw_tr      = sample_weight_all[tr]
            g_tr       = groups.iloc[tr]

            tr_mask_in, va_mask_in = make_inner_group_split(g_tr, inner_splits=INNER_SPLITS)
            # eval_set과 그에 맞는 가중치를 준비합니다.
            eval_set = [(X_tr[va_mask_in], f_t(y_tr[va_mask_in]))] if np.any(va_mask_in) else None
            # [추가] XGBoost는 eval 가중치를 리스트 형태로 받습니다.
            eval_wts = [sw_tr[va_mask_in]] if np.any(va_mask_in) else None

            model = build_model(params, n_estimators=5000, random_state=42)
            model.fit(
                X_tr[tr_mask_in], f_t(y_tr[tr_mask_in]),
                # 훈련 데이터에 대한 가중치
                sample_weight=sw_tr[tr_mask_in],
                # 검증 데이터 셋
                eval_set=eval_set,
                # [추가] 검증 데이터에 대한 가중치
                sample_weight_eval_set=eval_wts,
                verbose=False
            )

            bi = int(getattr(model, 'best_iteration', model.n_estimators)) + 1
            best_iters.append(bi)
            
            c = 1.0
            if TRANSFORM_KIND == 'log1p' and USE_SMEARING and np.any(va_mask_in):
                z_val = f_t(y_tr[va_mask_in]); z_hat = model.predict(X_tr[va_mask_in], iteration_range=(0, bi))
                resid = z_val - z_hat
                c = float(np.mean(np.exp(resid)))
                smear_factors.append(c)

            z_pred_te = model.predict(X_te, iteration_range=(0, bi))
            y_pred_te = finv_t(z_pred_te)
            if TRANSFORM_KIND == 'log1p' and USE_SMEARING:
                # OOF 생성 시에는 CV 전체에서 계산된 평균 스미어링 계수를 사용하는 것이 더 안정적일 수 있습니다.
                # 여기서는 폴드별 계수를 사용합니다.
                y_pred_te = c * (y_pred_te + 1.0) - 1.0

            oof_sum[te] += y_pred_te
            oof_cnt[te] += 1
            fold_mse.append(mean_squared_error(y_te, y_pred_te))
            fold_r2.append(r2_score(y_te, y_pred_te))

    valid = oof_cnt > 0
    oof_pred = np.zeros_like(oof_sum)
    oof_pred[valid] = oof_sum[valid] / oof_cnt[valid]
    
    stats = {
        "fold_r2_mean": np.mean(fold_r2), "fold_r2_std": np.std(fold_r2),
        "fold_mse_mean": np.mean(fold_mse), "fold_mse_std": np.std(fold_mse),
        "oof_r2": r2_score(y_all[valid], oof_pred[valid]),
        "oof_mse": mean_squared_error(y_all[valid], oof_pred[valid]),
        "best_iter_median": int(np.median(best_iters)) if best_iters else best_iter_from_search,
        "smearing_mean": np.mean(smear_factors) if smear_factors else c_cv_from_search
    }
    return oof_pred, stats, oof_cnt

oof_pred, oof_stats, oof_cnt = make_oof_with_params(best_params, n_runs=N_OOF_RUNS_BEST)
print("\n[BEST PARAMS OOF] "
      f"R^2={oof_stats['oof_r2']:.4f}, MSE={oof_stats['oof_mse']:.4f} | "
      f"fold R^2={oof_stats['fold_r2_mean']:.4f}±{oof_stats['fold_r2_std']:.4f}")

oof_cnt_is_positive = (oof_cnt > 0)

pd.DataFrame({
    'filename': df['filename'], 
    'seed_id': groups, 
    'y_true': y_all,
    'y_pred_oof': oof_pred,
    # Here, the boolean is converted to 1s and 0s for saving
    'has_oof': oof_cnt_is_positive.astype(int) 
}).to_csv(os.path.join(SAVE_DIR, "oof_predictions_best.csv"), index=False)

# ---------------------------
# 11) 최종 모델(전체 데이터) 재학습 + 저장
# ---------------------------
best_iter_final = int(oof_stats['best_iter_median'])
c_cv_final      = float(oof_stats['smearing_mean'])

print(f"\n[FINAL MODEL] Training with n_estimators={best_iter_final} and smearing_factor={c_cv_final:.4f}")
final_model_all = build_model(best_params, n_estimators=best_iter_final, random_state=42)
final_model_all.set_params(early_stopping_rounds=None) # 최종 학습에서는 조기종료 비활성화
final_model_all.fit(X_all, f_t(y_all), sample_weight=sample_weight_all)

# 모델 저장 (json 형식)
final_model_path = os.path.join(SAVE_DIR, "final_model.json")
final_model_all.save_model(final_model_path)
print(f"[SAVE] Final model saved to {final_model_path}")

# 전체 데이터 in-sample 예측 (보고/해석 참고용)
y_hat_all_t = final_model_all.predict(X_all)
y_hat_all   = finv_t(y_hat_all_t)
if TRANSFORM_KIND == 'log1p' and USE_SMEARING:
    y_hat_all = c_cv_final * (y_hat_all + 1.0) - 1.0

print(f"[INFO] In-sample R^2 = {r2_score(y_all, y_hat_all):.4f}")

In [ ]:
# Setting font configurations
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'
# ===================================================================
# 12) SHAP 안정성 검증 (여러 시드로 모델 재학습 후 평균 SHAP 계산)
# ===================================================================
print("\n[SHAP STABILITY] Starting SHAP stability analysis...")

# --- 1. 설정 ---
N_SHAP_RUNS = 100  # 안정성 검증을 위해 반복할 횟수
N_SHAP_SAMPLES = min(5000, X_all.shape[0]) # SHAP 계산에 사용할 샘플 수

# SHAP 계산에 사용할 데이터 샘플을 미리 고정 (통제된 실험)
rng_shap = np.random.default_rng(SEED_BASE)
idx_shap = rng_shap.choice(np.arange(X_all.shape[0]), size=N_SHAP_SAMPLES, replace=False)
X_shap_df = pd.DataFrame(X_all[idx_shap], columns=num_cols)

all_shap_values_sse = [] # 각 실행의 SSE 스케일 SHAP 값을 저장할 리스트

# --- 2. 반복 루프 시작 ---
for i in range(N_SHAP_RUNS):
    print(f"  > Running SHAP stability iteration {i+1}/{N_SHAP_RUNS}...")

    # A. 매번 다른 시드로 최종 모델을 다시 훈련
    model_for_shap_run = build_model(
        best_params,
        n_estimators=best_iter_final,
        random_state=SEED_BASE + i * 10  # 시드를 매번 변경하여 모델에 다양성 부여
    )
    model_for_shap_run.set_params(early_stopping_rounds=None)
    model_for_shap_run.fit(X_all, f_t(y_all), sample_weight=sample_weight_all)

    # B. 변환공간(z-space)에서의 SHAP 계산 (기존 로직과 동일)
    explainer_t = shap.TreeExplainer(model_for_shap_run)
    shap_z = explainer_t.shap_values(X_shap_df)
    z_pred = model_for_shap_run.predict(X_shap_df)

    # C. SSE 단위로 국소선형화 스케일 계산 (기존 로직과 동일)
    if TRANSFORM_KIND == 'log1p':
        scale = np.exp(z_pred)
    elif TRANSFORM_KIND == 'asinh':
        scale = np.cosh(z_pred)
    else: # 'raw'
        scale = np.ones_like(z_pred)
    
    if TRANSFORM_KIND == 'log1p' and USE_SMEARING:
        scale = c_cv_final * scale

    # D. SSE 스케일의 SHAP 값을 계산하여 리스트에 추가
    shap_sse_run = shap_z * scale.reshape(-1, 1)
    all_shap_values_sse.append(shap_sse_run)

# --- 3. 결과 집계 및 시각화 ---
print("[SHAP STABILITY] Aggregating results and plotting...")

# 모든 실행 결과의 평균 SHAP 값을 계산
mean_shap_values_sse = np.mean(all_shap_values_sse, axis=0)

# 평균 SHAP 값을 사용하여 최종 요약 플롯 생성
shap.summary_plot(mean_shap_values_sse, X_shap_df, show=False, cmap='jet', plot_type="dot", max_display=20)
plt.xlabel("Mean SHAP value (impact on SSE)")
plt.title("SHAP Feature Importance (Averaged over Runs)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_dot_mean_sse_scale.png"), dpi=160)
plt.close()

shap.summary_plot(mean_shap_values_sse, X_shap_df, show=False, plot_type="bar", max_display=20)
plt.xlabel("Mean absolute SHAP value in SSE units")
plt.title("SHAP Feature Importance (Averaged over Runs)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_bar_mean_sse_scale.png"), dpi=160)
plt.close()

print("[SAVE] Averaged SHAP summaries (SSE scale) saved.")


# ---------------------------
# 13) 요약/모델 저장
# ---------------------------
hp_summary = {
    "config": {
        "use_quick_features": bool(USE_QUICK_FEATURES),
        "transform": TRANSFORM_KIND,
        "use_smearing": bool(USE_SMEARING and smearable),
        "outer_splits": int(min(N_SPLITS, n_groups)),
        "inner_splits": int(INNER_SPLITS),
        "seed_base": int(SEED_BASE),
        "n_features": int(len(num_cols)),
        "n_samples": int(len(df)),
        "n_groups": int(n_groups),
        "hp_trials": int(N_HP_TRIALS),
        "hp_runs_per_trial": int(N_HP_RUNS),
        "oof_runs_best": int(N_OOF_RUNS_BEST)
    },
    "best_params": {
        "learning_rate": float(best_params["learning_rate"]),
        "max_depth": int(best_params["max_depth"]),
        "min_child_weight": int(best_params["min_child_weight"]),
        "subsample": float(best_params["subsample"]),
        "colsample_bytree": float(best_params["colsample_bytree"]),
        "reg_lambda": float(best_params["reg_lambda"]),
        "reg_alpha": float(best_params["reg_alpha"])
    },
    "search_best": {
        "r2_mean": float(best_row['r2_mean']),
        "r2_std":  float(best_row['r2_std']),
        "mse_mean": float(best_row['mse_mean']),
        "mse_std":  float(best_row['mse_std']),
        "best_iter_median": int(best_iter_from_search),
        "smearing_mean": float(c_cv_from_search)
    },
    "oof_stats_best": oof_stats,
    "final_model": {
        "n_estimators": int(best_iter_final),
        "smearing_factor_cv_mean": float(c_cv_final)
    }
}
with open(os.path.join(SAVE_DIR, "summary.json"), "w") as f:
    json.dump(hp_summary, f, indent=2)

final_model_all.save_model(os.path.join(SAVE_DIR, "final_model_all.json"))
print(f"[SAVE] outputs -> {SAVE_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

print("\n[VERIFICATION] Checking fold distribution of group SSEs...")

# 검증을 위해 StratifiedGroupKFold 객체를 하나 생성합니다.
sgkf_checker = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED_BASE)

# =================================================================
# [수정] 플롯을 위한 long-form 데이터프레임 생성
# =================================================================
# 각 폴드의 데이터를 담을 리스트
plot_data_list = []

for fold_idx, (train_indices, test_indices) in enumerate(sgkf_checker.split(X_all, y_stratify, groups=groups)):
    # 현재 폴드의 테스트 그룹(seed)들을 찾습니다.
    test_groups = groups.iloc[test_indices].unique()
    
    # 해당 그룹들의 '평균 log1p(SSE)' 값을 가져옵니다.
    current_fold_distribution = group_log_sse_means[test_groups]
    
    # 현재 폴드 데이터를 위한 작은 데이터프레임을 만듭니다.
    fold_df = pd.DataFrame({
        'fold': fold_idx,  # 폴드 번호 (x축으로 사용)
        'mean_log_sse': current_fold_distribution # SSE 값 (y축으로 사용)
    })
    
    plot_data_list.append(fold_df)

# 모든 폴드 데이터를 하나의 데이터프레임으로 합칩니다.
plot_df_long = pd.concat(plot_data_list, ignore_index=True)
# =================================================================

# 1. 시각화: 박스 플롯으로 분포 확인
plt.figure(figsize=(12, 7))
# [수정] x축과 y축, 데이터를 명확하게 지정해줍니다.
sns.boxplot(x='fold', y='mean_log_sse', data=plot_df_long)
plt.title('Distribution of Group-Mean log1p(SSE) Across Folds', fontsize=16)
plt.xlabel('Fold Index', fontsize=12)
plt.ylabel('Group-Mean log1p(SSE)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


# 2. 통계 요약: DataFrame으로 확인 (이전과 동일하게 작동)
summary_stats = plot_df_long.groupby('fold')['mean_log_sse'].describe()

print("\nSummary Statistics of Group-Mean log1p(SSE) for each fold's test set:")
print(summary_stats[['count', 'mean', 'std', 'min', 'max']])

In [ ]:
# Setting font configurations
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'
# ===================================================================
# 12) SHAP 안정성 검증 (여러 시드로 모델 재학습 후 평균 SHAP 계산)
# ===================================================================
print("\n[SHAP STABILITY] Starting SHAP stability analysis...")

# --- 1. 설정 ---
N_SHAP_RUNS = 100  # 안정성 검증을 위해 반복할 횟수
N_SHAP_SAMPLES = min(5000, X_all.shape[0]) # SHAP 계산에 사용할 샘플 수

# SHAP 계산에 사용할 데이터 샘플을 미리 고정 (통제된 실험)
rng_shap = np.random.default_rng(SEED_BASE)
idx_shap = rng_shap.choice(np.arange(X_all.shape[0]), size=N_SHAP_SAMPLES, replace=False)
X_shap_df = pd.DataFrame(X_all[idx_shap], columns=num_cols)

all_shap_values_sse = [] # 각 실행의 SSE 스케일 SHAP 값을 저장할 리스트

# --- 2. 반복 루프 시작 ---
for i in range(N_SHAP_RUNS):
    print(f"  > Running SHAP stability iteration {i+1}/{N_SHAP_RUNS}...")

    # A. 매번 다른 시드로 최종 모델을 다시 훈련
    model_for_shap_run = build_model(
        best_params,
        n_estimators=best_iter_final,
        random_state=SEED_BASE + i * 10  # 시드를 매번 변경하여 모델에 다양성 부여
    )
    model_for_shap_run.set_params(early_stopping_rounds=None)
    model_for_shap_run.fit(X_all, f_t(y_all), sample_weight=sample_weight_all)

    # B. 변환공간(z-space)에서의 SHAP 계산 (기존 로직과 동일)
    explainer_t = shap.TreeExplainer(model_for_shap_run)
    shap_z = explainer_t.shap_values(X_shap_df)
    z_pred = model_for_shap_run.predict(X_shap_df)

    # C. SSE 단위로 국소선형화 스케일 계산 (기존 로직과 동일)
    if TRANSFORM_KIND == 'log1p':
        scale = np.exp(z_pred)
    elif TRANSFORM_KIND == 'asinh':
        scale = np.cosh(z_pred)
    else: # 'raw'
        scale = np.ones_like(z_pred)
    
    if TRANSFORM_KIND == 'log1p' and USE_SMEARING:
        scale = c_cv_final * scale

    # D. SSE 스케일의 SHAP 값을 계산하여 리스트에 추가
    shap_sse_run = shap_z * scale.reshape(-1, 1)
    all_shap_values_sse.append(shap_sse_run)

# --- 3. 결과 집계 및 시각화 ---
print("[SHAP STABILITY] Aggregating results and plotting...")

# 모든 실행 결과의 평균 SHAP 값을 계산
mean_shap_values_sse = np.mean(all_shap_values_sse, axis=0)

# 평균 SHAP 값을 사용하여 최종 요약 플롯 생성
shap.summary_plot(mean_shap_values_sse, X_shap_df, show=False, cmap='jet', plot_type="dot", max_display=20)
plt.xlabel("Mean SHAP value (impact on SSE)")
plt.title("SHAP Feature Importance (Averaged over Runs)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_dot_mean_sse_scale.png"), dpi=160)
plt.close()

shap.summary_plot(mean_shap_values_sse, X_shap_df, show=False, plot_type="bar", max_display=20)
plt.xlabel("Mean absolute SHAP value in SSE units")
plt.title("SHAP Feature Importance (Averaged over Runs)")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_bar_mean_sse_scale.png"), dpi=160)
plt.close()

print("[SAVE] Averaged SHAP summaries (SSE scale) saved.")


# ---------------------------
# 13) 요약/모델 저장
# ---------------------------
hp_summary = {
    "config": {
        "use_quick_features": bool(USE_QUICK_FEATURES),
        "transform": TRANSFORM_KIND,
        "use_smearing": bool(USE_SMEARING and smearable),
        "outer_splits": int(min(N_SPLITS, n_groups)),
        "inner_splits": int(INNER_SPLITS),
        "seed_base": int(SEED_BASE),
        "n_features": int(len(num_cols)),
        "n_samples": int(len(df)),
        "n_groups": int(n_groups),
        "hp_trials": int(N_HP_TRIALS),
        "hp_runs_per_trial": int(N_HP_RUNS),
        "oof_runs_best": int(N_OOF_RUNS_BEST)
    },
    "best_params": {
        "learning_rate": float(best_params["learning_rate"]),
        "max_depth": int(best_params["max_depth"]),
        "min_child_weight": int(best_params["min_child_weight"]),
        "subsample": float(best_params["subsample"]),
        "colsample_bytree": float(best_params["colsample_bytree"]),
        "reg_lambda": float(best_params["reg_lambda"]),
        "reg_alpha": float(best_params["reg_alpha"])
    },
    "search_best": {
        "r2_mean": float(best_row['r2_mean']),
        "r2_std":  float(best_row['r2_std']),
        "mse_mean": float(best_row['mse_mean']),
        "mse_std":  float(best_row['mse_std']),
        "best_iter_median": int(best_iter_from_search),
        "smearing_mean": float(c_cv_from_search)
    },
    "oof_stats_best": oof_stats,
    "final_model": {
        "n_estimators": int(best_iter_final),
        "smearing_factor_cv_mean": float(c_cv_final)
    }
}
with open(os.path.join(SAVE_DIR, "summary.json"), "w") as f:
    json.dump(hp_summary, f, indent=2)

final_model_all.save_model(os.path.join(SAVE_DIR, "final_model_all.json"))
print(f"[SAVE] outputs -> {SAVE_DIR}")

In [ ]:
df.columns

In [ ]:
import optuna
import pandas as pd
import numpy as np
import xgboost as xgb
import json

# --- 1. 사전 준비: 모델, 특성 순서, 변환 함수 로딩 ---

# 저장된 최종 모델 불러오기
final_model = final_model_all

# ⚠️ 매우 중요: 모델이 학습할 때 사용했던 '정확한' 특성 순서 리스트
# 이 순서가 다르면 예측이 완전히 잘못됩니다.
# 사용자의 summary.json이나 학습 코드에서 이 리스트를 가져와야 합니다.
# 아래는 예시 리스트이므로, 반드시 본인의 것으로 교체해주세요.
FEATURE_ORDER = [
    'M_X_avg_bond_length', 'M_X_max_bond_length',
    'M_X_min_bond_length', 'M_X_std_bond_length', 'XMX_max_angle',
    'XMX_min_angle', 'XMX_avg_angle', 'XMX_std_angle', 'XXX_max_angle',
    'XXX_min_angle', 'XXX_std_angle', 'inter_motif_dist_1',
    'inter_motif_dist_2', 'inter_motif_dist_3', 'M_M_dist_1', 'M_M_dist_2',
    'M_M_dist_3', 'motif_long_axis', 'motif_short_axis', 'motif_axis_ratio',
    'num_X_in_motif', 'Z_M', 'M_electronegativity', 'Z_X',
    'X_electronegativity', 'motif_alignment_angle', 'motif_volume',
    'motif_surface_area', 'motif_unitcell_vol_ratio', 'min_match_rmsd',
    'min_match_rmsd_std', 'M_d_nelec', 'X_p_nelec', 'delta_chi',
    'abs_delta_chi', 'delta_Z', 'abs_delta_Z', 'p_d_elec_ratio',
    'max_avg_len_diff', 'bond_len_range', 'bond_len_cv', 'XMX_angle_range',
    'XXX_angle_range', 'abs_delta_chi_x_len_diff', 'mag_inter_dist_diff_1',
    'mag_inter_dist_diff_2', 'mag_inter_dist_diff_3', 'packing_density',
    'proxy_M_magnet'
]


# 모델 훈련 시 사용했던 타겟 변환 함수 (사용자 코드 기반)
TRANSFORM_KIND = 'log1p' # 'asinh', 'raw' 등 실제 사용했던 것으로 변경

def f_inv(z):
    if TRANSFORM_KIND == 'log1p':
        return np.expm1(z)
    elif TRANSFORM_KIND == 'asinh':
        return np.sinh(z)
    return z

electronegativity_map = {21: 1.36, 22: 1.54, 23: 1.63, 24: 1.66, 25: 1.55, 26: 1.83, 27: 1.88, 28: 1.91, 29: 1.90, 30: 10, 39: 1.22, 40: 1.33, 41: 1.6, 42: 2.15, 44: 2.2, 45: 2.28, 46: 2.2, 47: 1.93, 48: 1.69,
                         5: 2.04, 6: 2.55, 7: 3.04, 8: 3.44, 9: 3.98, 13: 1.61, 14: 1.9, 15: 2.19, 16: 2.68, 17: 3.16, 31: 1.81, 32: 2.01, 33: 2.18, 34: 2.55, 35: 2.95, 49: 1.78, 50: 1.96, 51: 2.05, 52: 2.1, 53: 2.65, 81: 1.62, 82: 2.33, 83: 2.02}

d_electron_map  = {21: 2, 22: 3, 23: 4, 24: 5 , 25: 6, 26: 7, 27: 8, 28: 9, 29: 10, 30: 10, 39: 2, 40: 3, 41: 4, 42: 5, 44: 7, 45: 8, 46: 9, 47: 10, 48: 10}

p_electron_map = {5: 1, 6: 2, 7: 3, 8: 4, 9: 5, 13: 1, 14: 2, 15: 3, 16: 4, 17: 5, 31: 1, 32: 2, 33: 3, 34: 4, 35: 5, 49: 1,50: 2, 51: 3, 52: 4, 53: 5, 81: 1, 82: 2, 83: 3}

proxy_map = {21:0, 22:1.73, 23:2.83, 24: 3.87, 25: 4.90, 26: 4.90, 27: 3.87, 28: 2.83, 29: 1.73, 30: 0, 39:0, 40:1.73, 41: 2.83, 42: 3.87, 44: 4.90, 45: 3.87, 46: 2.83, 47: 1.73, 48:0}

# --- 2. Optuna Objective 함수 정의 ---
# 이 함수는 "가상의 물질" 특성 조합을 만들어 모델에게 SSE를 물어보는 역할을 합니다.

def objective(trial):
    """최적의 특성 조합을 찾는 목표 함수 (수정된 버전)"""
    
    features = {}

    # ===================================================================
    # 1. 독립적인 기본 특성 샘플링 (Independent Base Features)
    # ===================================================================
    
    # 1-1. 원자 구성
    z_m = trial.suggest_categorical('Z_M', [24, 25, 26, 27, 28, 29])
    z_x = trial.suggest_categorical('Z_X', [7, 8, 15, 16, 33, 34, 51, 52, 83])
    
    # 1-2. 기본 기하학적 특성
    avg_len = trial.suggest_float('M_X_avg_bond_length', 1.8, 2.8)
    delta_pos = trial.suggest_float('delta_pos_len', 0.0, 0.5)
    delta_neg = trial.suggest_float('delta_neg_len', 0.0, 0.5)
    
    xmx_min_angle = trial.suggest_float('XMX_min_angle', 80.0, 179.0)
    xmx_angle_range = trial.suggest_float('XMX_angle_range', 1.0, 50.0)
    xxx_min_angle = trial.suggest_float('XXX_min_angle', 80.0, 120.0)
    xxx_angle_range = trial.suggest_float('XXX_angle_range', 1.0, 40.0)
    
    inter_d1 = trial.suggest_float('inter_motif_dist_1', 2.5, 4.0)
    mm_d1 = trial.suggest_float('M_M_dist_1', 2.5, 4.0)
    
    axis_ratio = trial.suggest_float('motif_axis_ratio', 0.5, 1.0)
    
    motif_vol = trial.suggest_float('motif_volume', 20.0, 150.0)
    rmsd = trial.suggest_float('min_match_rmsd', 0.0, 3.0)
    align_angle = trial.suggest_float('motif_alignment_angle', 0.0, 180.0)
    
    # ✨ FIX: 이 줄을 추가하여 누락된 특성을 포함합니다.
    features['max_avg_len_diff'] = trial.suggest_float('max_avg_len_diff', 0.0, 0.5)

    # ===================================================================
    # 2. 종속적인 파생 특성 계산 (Dependent Derived Features)
    # ===================================================================

    # 2-1. 하드코딩된 값 및 원자 속성
    features['num_X_in_motif'] = 6
    features.update({
        'Z_M': z_m, 'Z_X': z_x,
        'M_electronegativity': electronegativity_map[z_m],
        'X_electronegativity': electronegativity_map[z_x],
        'M_d_nelec': d_electron_map[z_m],
        'X_p_nelec': p_electron_map[z_x]
    })
    
    # 2-2. 결합 길이 상세
    max_len = avg_len + delta_pos
    min_len = avg_len - delta_neg
    if min_len <= 0: raise optuna.exceptions.TrialPruned()
    bond_range = max_len - min_len
    std_len = bond_range / 4.0
    features.update({
        'M_X_avg_bond_length': avg_len, 'M_X_max_bond_length': max_len,
        'M_X_min_bond_length': min_len, 'M_X_std_bond_length': std_len,
        'bond_len_range': bond_range, 'bond_len_cv': std_len / avg_len if avg_len > 1e-6 else 0
    })

    # 2-3. 각도 상세
    xmx_max_angle = xmx_min_angle + xmx_angle_range
    if xmx_max_angle > 180.0: raise optuna.exceptions.TrialPruned()
    features.update({
        'XMX_max_angle': xmx_max_angle, 'XMX_min_angle': xmx_min_angle,
        'XMX_avg_angle': (xmx_min_angle + xmx_max_angle) / 2.0,
        'XMX_std_angle': xmx_angle_range / 4.0, 'XMX_angle_range': xmx_angle_range
    })
    
    xxx_max_angle = xxx_min_angle + xxx_angle_range
    if xxx_max_angle > 180.0: raise optuna.exceptions.TrialPruned()
    features.update({
        'XXX_max_angle': xxx_max_angle, 'XXX_min_angle': xxx_min_angle,
        'XXX_std_angle': xxx_angle_range / 4.0, 'XXX_angle_range': xxx_angle_range
    })

    # 2-4. 거리 상세
    inter_d2 = inter_d1 + trial.suggest_float('delta_inter_2', 0.01, 1.0)
    inter_d3 = inter_d2 + trial.suggest_float('delta_inter_3', 0.01, 1.0)
    mm_d2 = mm_d1 + trial.suggest_float('delta_mm_2', 0.01, 1.0)
    mm_d3 = mm_d2 + trial.suggest_float('delta_mm_3', 0.01, 1.0)
    features.update({
        'inter_motif_dist_1': inter_d1, 'inter_motif_dist_2': inter_d2, 'inter_motif_dist_3': inter_d3,
        'M_M_dist_1': mm_d1, 'M_M_dist_2': mm_d2, 'M_M_dist_3': mm_d3
    })
    
    # 2-5. 모티프/구조 상세
    long_axis = 2.0 * max_len
    short_axis = long_axis * axis_ratio
    unit_cell_vol = motif_vol * trial.suggest_float('cell_to_motif_ratio', 1.1, 5.0)
    surface_area = trial.suggest_float('surface_area_prefactor', 4.5, 6.0) * (motif_vol ** (2/3))
    rmsd_std = rmsd * trial.suggest_float('rmsd_std_fraction', 0.1, 0.5)
    
    features.update({
        'motif_long_axis': long_axis, 'motif_short_axis': short_axis, 'motif_axis_ratio': axis_ratio,
        'motif_volume': motif_vol, 'min_match_rmsd': rmsd, 'motif_alignment_angle': align_angle,
        'motif_surface_area': surface_area, 'min_match_rmsd_std': rmsd_std,
        'motif_unitcell_vol_ratio': motif_vol / unit_cell_vol,
        'packing_density': trial.suggest_float('packing_density', 10, 40)
    })
    
    # 2-6. 최종 상호작용 및 차이 특성
    delta_z = float(z_m - z_x)
    delta_chi = features['M_electronegativity'] - features['X_electronegativity']
    features.update({
        'delta_Z': delta_z, 'abs_delta_Z': abs(delta_z),
        'delta_chi': delta_chi, 'abs_delta_chi': abs(delta_chi),
        'p_d_elec_ratio': features['X_p_nelec'] / (features['M_d_nelec'] + 1e-6)
    })
    
    features['proxy_M_magnet'] = proxy_map[z_m]
    features['abs_delta_chi_x_len_diff'] = abs(delta_chi) * features['max_avg_len_diff']
    features['mag_inter_dist_diff_1'] = features['M_M_dist_1'] - features['inter_motif_dist_1']
    features['mag_inter_dist_diff_2'] = features['M_M_dist_2'] - features['inter_motif_dist_2']
    features['mag_inter_dist_diff_3'] = features['M_M_dist_3'] - features['inter_motif_dist_3']

    # ===================================================================
    # 3. 최종 검증 및 예측
    # ===================================================================
    if len(features) != len(FEATURE_ORDER):
        missing = set(FEATURE_ORDER) - set(features.keys())
        raise ValueError(f"Feature mismatch! {len(features)} defined, {len(FEATURE_ORDER)} required. Missing: {missing}")

    input_df = pd.DataFrame([features])[FEATURE_ORDER]
    prediction_t = final_model.predict(input_df)[0]
    predicted_sse = f_inv(prediction_t)
    
    return predicted_sse

# --- 3. 베이지안 최적화 실행 ---

print("[BO] Starting Feature Optimization...")

# SSE를 '최대화(maximize)'하는 것이 목표
study = optuna.create_study(direction='maximize')

# 최적화 실행
study.optimize(
    objective,
    n_trials=5, # 충분한 횟수로 탐색 (더 많이 할수록 좋음)
    show_progress_bar=False
)

# --- 4. 최종 결과 출력 ---

print("\n[BO] Optimization Finished!")
print(f"  > Model's Predicted Maximum SSE: {study.best_value:.6f}")
print("  > Optimal Feature Combination Found:")
for key, value in study.best_params.items():
    print(f"    - {key}: {value}")

In [ ]:
import optuna
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA

# ===================================================================
# 1. 사전 준비 (Setup)
# ===================================================================

# ⚠️ TODO: 이 부분들을 사용자의 환경에 맞게 설정하세요.
# final_model = xgb.XGBRegressor()
# final_model.load_model("final_model_all.json")
# FEATURE_ORDER = [...] # 모델이 학습한 정확한 특성 순서 리스트

# --- 룩업 테이블 (Lookup Tables) ---
electronegativity_map = {
    7: 3.04, 8: 3.44, 15: 2.19, 16: 2.58, 24: 1.66, 25: 1.55, 26: 1.83, 27: 1.88, 28: 1.91, 29: 1.90,
    33: 2.18, 34: 2.55, 51: 2.05, 52: 2.1, 83: 2.02
}
d_electron_map = {24: 5, 25: 5, 26: 6, 27: 7, 28: 8, 29: 10}
p_electron_map = {7: 3, 8: 4, 15: 3, 16: 4, 33: 3, 34: 4, 51: 3, 52: 4, 83: 3}
proxy_map = {24: 3.87, 25: 4.90, 26: 4.90, 27: 3.87, 28: 2.83, 29: 1.73}

# --- 타겟 역변환 함수 ---
TRANSFORM_KIND = 'log1p'
def f_inv(z):
    if TRANSFORM_KIND == 'log1p': return np.expm1(z)
    if TRANSFORM_KIND == 'asinh': return np.sinh(z)
    return z

# ===================================================================
# 2. 핵심 헬퍼 함수 (Core Helper Function) ⚛️
# ===================================================================
import numpy as np
from scipy.spatial import ConvexHull
from sklearn.decomposition import PCA

def generate_and_calculate_geometry(trial):
    """
    Optuna trial로부터 기본 파라미터를 받아 6개의 원자 좌표를 생성하고,
    관련된 '모든' 기하학적 특성을 계산하여 딕셔너리로 반환합니다.
    """
    # --- 1. 6개의 비자성 원자(X) 좌표 생성 (기존과 동일) ---
    coords_cartesian = []
    bond_lengths = []
    for i in range(6):
        r = trial.suggest_float(f'r_{i}', 1.8, 2.8)
        theta_deg = trial.suggest_float(f'theta_{i}', 0, 180)
        phi_deg = trial.suggest_float(f'phi_{i}', 0, 360)
        
        theta_rad, phi_rad = np.deg2rad(theta_deg), np.deg2rad(phi_deg)
        x = r * np.sin(theta_rad) * np.cos(phi_rad)
        y = r * np.sin(theta_rad) * np.sin(phi_rad)
        z = r * np.cos(theta_rad)
        
        coords_cartesian.append([x, y, z])
        bond_lengths.append(r)
        
    coords = np.array(coords_cartesian)
    
    # --- 2. 생성된 좌표로부터 모든 기하학적 특성 계산 ---
    
    # A. 결합 길이 통계 (기존과 동일)
    bond_stats = {
        'M_X_avg_bond_length': np.mean(bond_lengths), 'M_X_max_bond_length': np.max(bond_lengths),
        'M_X_min_bond_length': np.min(bond_lengths), 'M_X_std_bond_length': np.std(bond_lengths)
    }
    bond_stats['bond_len_range'] = bond_stats['M_X_max_bond_length'] - bond_stats['M_X_min_bond_length']
    bond_stats['bond_len_cv'] = bond_stats['M_X_std_bond_length'] / bond_stats['M_X_avg_bond_length']

    # B. X-M-X 각도 통계 (기존과 동일)
    xmx_angles = []
    for i in range(6):
        for j in range(i + 1, 6):
            v1, v2 = coords[i], coords[j]
            cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
            xmx_angles.append(np.rad2deg(np.arccos(np.clip(cos_angle, -1.0, 1.0))))
    
    xmx_stats = {
        'XMX_avg_angle': np.mean(xmx_angles), 'XMX_std_angle': np.std(xmx_angles),
        'XMX_min_angle': np.min(xmx_angles), 'XMX_max_angle': np.max(xmx_angles),
        'XMX_angle_range': np.max(xmx_angles) - np.min(xmx_angles)
    }

    # C. ✨ XXX 각도 통계 계산 (새로 추가된 부분) ✨
    xxx_angles = []
    for j in range(6): # j가 중심 원자
        for i in range(6):
            for k in range(i + 1, 6):
                # i-j-k 각도를 계산. i, j, k는 모두 달라야 함.
                if i == j or k == j:
                    continue
                
                p_i, p_j, p_k = coords[i], coords[j], coords[k]
                v_ji = p_i - p_j # j -> i 벡터
                v_jk = p_k - p_j # j -> k 벡터
                
                cos_angle = np.dot(v_ji, v_jk) / (np.linalg.norm(v_ji) * np.linalg.norm(v_jk))
                angle = np.rad2deg(np.arccos(np.clip(cos_angle, -1.0, 1.0)))
                xxx_angles.append(angle)

    xxx_stats = {
        'XXX_max_angle': np.max(xxx_angles),
        'XXX_min_angle': np.min(xxx_angles),
        'XXX_std_angle': np.std(xxx_angles),
        'XXX_angle_range': np.max(xxx_angles) - np.min(xxx_angles)
    }
    
    # D. 모티프 부피, 표면적, 축 계산 (기존과 동일)
    hull = ConvexHull(coords)
    motif_stats = {'motif_volume': hull.volume, 'motif_surface_area': hull.area}
    
    pca = PCA(n_components=3).fit(coords)
    axis_lengths = np.sqrt(pca.explained_variance_)
    motif_stats['motif_long_axis'] = axis_lengths[0] * 2
    motif_stats['motif_short_axis'] = axis_lengths[1] * 2
    motif_stats['motif_axis_ratio'] = axis_lengths[1] / axis_lengths[0]
    
    # 모든 계산된 특성들을 하나의 딕셔너리로 통합하여 반환
    return {**bond_stats, **xmx_stats, **xxx_stats, **motif_stats}

# ===================================================================
# 3. 최종 Objective 함수 및 실행 (Execution) 🚀
# ===================================================================
def objective(trial):
    """최적의 특성 조합을 찾는 목표 함수 (최종 완성 버전)"""
    features = {}

    # 1. 원자 구성 및 소수의 독립 변수 샘플링
    z_m = trial.suggest_categorical('Z_M', [24, 25, 26, 27, 28, 29])
    z_x = trial.suggest_categorical('Z_X', [7, 8, 15, 16, 33, 34, 51, 52, 83])

    # 2. 헬퍼 함수를 호출하여 '모든' 기하학적 특성을 한 번에 계산
    geom_features = generate_and_calculate_geometry(trial)
    features.update(geom_features)
    
    # 3. 나머지 모든 종속 특성 계산
    features.update({
        'Z_M': z_m, 'Z_X': z_x, 'num_X_in_motif': 6,
        'M_electronegativity': electronegativity_map[z_m], 'X_electronegativity': electronegativity_map[z_x],
        'M_d_nelec': d_electron_map[z_m], 'X_p_nelec': p_electron_map[z_x]
    })
    
    features['min_match_rmsd'] = trial.suggest_float('min_match_rmsd', 0.0, 1.0)
    features['motif_alignment_angle'] = trial.suggest_float('motif_alignment_angle', 0.0, 90.0)
    features['max_avg_len_diff'] = trial.suggest_float('max_avg_len_diff', 0.0, 0.5)
    
    unit_cell_vol = features['motif_volume'] * trial.suggest_float('cell_to_motif_ratio', 1.1, 5.0)
    features.update({
        'min_match_rmsd_std': features['min_match_rmsd'] * trial.suggest_float('rmsd_std_fraction', 0.1, 0.5),
        'motif_unitcell_vol_ratio': features['motif_volume'] / unit_cell_vol,
        'packing_density': trial.suggest_float('packing_density', 0.3, 0.75)
    })
    
    delta_z = float(z_m - z_x)
    delta_chi = features['M_electronegativity'] - features['X_electronegativity']
    features.update({
        'delta_Z': delta_z, 'abs_delta_Z': abs(delta_z), 'delta_chi': delta_chi,
        'abs_delta_chi': abs(delta_chi), 'p_d_elec_ratio': features['X_p_nelec'] / (features['M_d_nelec'] + 1e-6)
    })
    
    features['proxy_M_magnet'] = proxy_map[z_m]
    features['abs_delta_chi_x_len_diff'] = abs(delta_chi) * features['max_avg_len_diff']
    
    inter_d1 = trial.suggest_float('inter_motif_dist_1', 2.5, 4.0)
    inter_d2 = inter_d1 + trial.suggest_float('delta_inter_2', 0.01, 1.0)
    inter_d3 = inter_d2 + trial.suggest_float('delta_inter_3', 0.01, 1.0)
    mm_d1 = trial.suggest_float('M_M_dist_1', 2.5, 4.0)
    mm_d2 = mm_d1 + trial.suggest_float('delta_mm_2', 0.01, 1.0)
    mm_d3 = mm_d2 + trial.suggest_float('delta_mm_3', 0.01, 1.0)
    features.update({
        'inter_motif_dist_1': inter_d1, 'inter_motif_dist_2': inter_d2, 'inter_motif_dist_3': inter_d3,
        'M_M_dist_1': mm_d1, 'M_M_dist_2': mm_d2, 'M_M_dist_3': mm_d3,
        'mag_inter_dist_diff_1': mm_d1 - inter_d1, 'mag_inter_dist_diff_2': mm_d2 - inter_d2,
        'mag_inter_dist_diff_3': mm_d3 - inter_d3
    })
    
    # 4. 최종 예측
    input_df = pd.DataFrame([features])[FEATURE_ORDER]
    prediction_t = final_model.predict(input_df)[0]
    predicted_sse = f_inv(prediction_t)
    
    return predicted_sse

# # --- BO 실행 (아래 주석을 해제하여 실행) ---
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5000)

# # --- 최종 결과 출력 ---
print("\n[BO] Optimization Finished!")
print(f"  > Model's Predicted Maximum SSE: {study.best_value:.6f}")
print("  > Optimal Independent Parameters Found:")
for key, value in study.best_params.items():
    print(f"    - {key}: {value}")

In [ ]:
# ============================================================
# Inverse Design via Anchor-based Feature Optimization (CEM)
#  - pick anchor rows (real data) → vary only chosen design variables
#  - recompute quick features to keep internal consistency
#  - maximize model-predicted SSE
# ============================================================
import os, json
import numpy as np
import pandas as pd

SAVE_DIR_INV = "20251020_inverse_design_anchor_opt_BO"
os.makedirs(SAVE_DIR_INV, exist_ok=True)

# ---- 헬퍼: 모델 예측 (log1p+smear 대응) ----
def predict_sse_from_matrix(X_mat, model, transform_kind='log1p', use_smear=False, smear_factor=1.0):
    z = model.predict(X_mat)
    if transform_kind == 'log1p':
        y = np.expm1(z)
        if use_smear:
            y = smear_factor * (y + 1.0) - 1.0
        return y
    elif transform_kind == 'asinh':
        return np.sinh(z)
    else:
        return z

# ---- 파생 피처 재계산 (질문에서 제공한 함수와 동일) ----
def add_quick_features(df_):
    df_ = df_.copy()
    def has(c): return c in df_.columns

    if has('magnetic_electronegativity') and has('nonmagnetic_electronegativity'):
        df_['delta_chi'] = df_['magnetic_electronegativity'] - df_['nonmagnetic_electronegativity']
        df_['abs_delta_chi'] = df_['delta_chi'].abs()
    if has('magnetic_atomic_number') and has('nonmagnetic_atomic_number'):
        df_['delta_Z'] = df_['magnetic_atomic_number'] - df_['nonmagnetic_atomic_number']
        df_['abs_delta_Z'] = df_['delta_Z'].abs()
    if has('mag_charge') and has('nonmag_charge'):
        df_['delta_q'] = df_['mag_charge'] - df_['nonmag_charge']
        df_['abs_delta_q'] = df_['delta_q'].abs()
    if has('p_orb_e_non') and has('d_orb_e'):
        df_['pd_ratio'] = df_['p_orb_e_non'] / (df_['d_orb_e'] + 1e-6)
    if all(has(c) for c in ['max_bond_length','avg_bond_length','min_bond_length','std_bond_length']):
        df_['ax_eq_gap']  = df_['max_bond_length'] - df_['avg_bond_length']
        df_['bond_range'] = df_['max_bond_length'] - df_['min_bond_length']
        df_['bond_cv']    = df_['std_bond_length'] / (df_['avg_bond_length'] + 1e-6)
    if has('center_max_angle') and has('center_min_angle'):
        df_['center_angle_spread'] = df_['center_max_angle'] - df_['center_min_angle']
    if has('nonmag_max_angle') and has('nonmag_min_angle'):
        df_['nonmag_angle_spread'] = df_['nonmag_max_angle'] - df_['nonmag_min_angle']
    if has('avg_hull_volume') and has('avg_hull_area'):
        df_['hull_sphericity'] = (np.pi**(1/3) * (6.0*df_['avg_hull_volume'])**(2/3)) / (df_['avg_hull_area'] + 1e-9)
    if has('rotation_angle_deg'):
        rad = np.deg2rad(df_['rotation_angle_deg'])
        df_['rot_sin'] = np.sin(rad)
        df_['rot_cos'] = np.cos(rad)
    if has('hub_U') and 'ax_eq_gap' in df_.columns:
        df_['U_times_axeq'] = df_['hub_U'] * df_['ax_eq_gap']
    if 'abs_delta_chi' in df_.columns and 'ax_eq_gap' in df_.columns:
        df_['delta_chi_times_axeq'] = df_['abs_delta_chi'] * df_['ax_eq_gap']

    for k in ['1st','2nd','3rd']:
        if has(f'global_{k}') and has(f'labelled_{k}'):
            df_[f'd_global_local_{k}'] = df_[f'global_{k}'] - df_[f'labelled_{k}']
    return df_

# ---- 설계변수 사양: 바꿀 컬럼과 경계 설정 ----
# 데이터 분위수 기반 경계(1%~99%) + 앵커 기준 ±범위 혼합
ALL_NUM_COLS = num_cols  # 기존 파이프라인에서 정의됨

# 설계변수 후보 (필요시 추가/제외)
DESIGN_VARS = [
    'avg_bond_length','max_bond_length','min_bond_length','std_bond_length',
    'center_max_angle','center_min_angle','nonmag_max_angle','nonmag_min_angle',
    'rotation_angle_deg','avg_axis_ratio'
]

# 정수형/이산형이 아닌 **연속 변수** 위주로 선택하세요.
DESIGN_VARS = [c for c in DESIGN_VARS if c in ALL_NUM_COLS]

# 경계 도출
q_lo, q_hi = 0.01, 0.99
bounds = {}
for c in DESIGN_VARS:
    lo = float(df[c].quantile(q_lo))
    hi = float(df[c].quantile(q_hi))
    if lo == hi:  # 변동이 없을 때 안전장치
        lo, hi = float(df[c].min()), float(df[c].max())
    bounds[c] = (lo, hi)

# ---- CEM 최적화 본체 ----
def cem_optimize_anchor(anchor_row: pd.Series,
                        model,
                        design_vars: list,
                        bounds: dict,
                        all_num_cols: list,
                        transform_kind='log1p',
                        use_smear=False,
                        smear_factor=1.0,
                        n_iter=12, pop=512, elite_frac=0.12, shrink=0.7, seed=2025):
    """
    anchor_row: 실제 데이터의 한 행 (df.loc[idx])
    design_vars: 조절할 연속 변수 리스트
    bounds: 각 변수의 (lo,hi)
    shrink: 매 반복마다 표준편차를 줄이는 계수(0.7~0.9)
    반환: 최고 후보 N개 DataFrame
    """
    rng = np.random.default_rng(seed)
    # 초기 분포: 앵커 중심, 데이터 경계 안에서 표준편차 설정
    mu = np.array([float(np.clip(anchor_row[v], *bounds[v])) for v in design_vars], dtype=float)
    sigma = np.array([max(1e-6, (bounds[v][1]-bounds[v][0]) * 0.15) for v in design_vars], dtype=float)

    records = []
    for it in range(n_iter):
        # 표본 생성 (경계 내 트렁케이션)
        Xvar = []
        for _ in range(pop):
            s = rng.normal(mu, sigma)
            # 경계 클리핑
            for j,v in enumerate(design_vars):
                lo, hi = bounds[v]
                s[j] = np.clip(s[j], lo, hi)
            Xvar.append(s)
        Xvar = np.vstack(Xvar)  # [pop, n_var]

        # 앵커 기반 전체 피처 행 구축 + 파생 피처 재계산
        batch_rows = []
        for i in range(pop):
            row = anchor_row.copy()
            for j,v in enumerate(design_vars):
                row[v] = Xvar[i, j]
            df1 = pd.DataFrame([row])
            df1 = add_quick_features(df1)
            # 모델 입력 컬럼만 추출(부족하면 앵커값 유지)
            feat = df1[all_num_cols].values
            batch_rows.append(feat[0])
        X_mat = np.vstack(batch_rows)  # [pop, n_feat]

        # 예측 및 순위
        y_pred = predict_sse_from_matrix(X_mat, model, transform_kind, use_smear, smear_factor)
        elite_k = max(5, int(pop * elite_frac))
        elite_idx = np.argsort(-y_pred)[:elite_k]

        # 분포 업데이트
        elite_vars = Xvar[elite_idx]
        mu = elite_vars.mean(axis=0)
        sigma = elite_vars.std(axis=0) * shrink
        sigma = np.maximum(sigma, 1e-6)

        # 로깅
        best_idx = elite_idx[0]
        records.append({
            "iter": it+1,
            "best_pred": float(y_pred[best_idx]),
            **{f"best_{v}": float(Xvar[best_idx, j]) for j,v in enumerate(design_vars)}
        })
        print(f"[CEM it {it+1}/{n_iter}] best_pred={y_pred[best_idx]:.4f}")

    # 마지막 세대에서 상위 N 추출
    N_KEEP = 100
    keep_idx = np.argsort(-y_pred)[:N_KEEP]
    keep_rows = []
    for k in keep_idx:
        row = anchor_row.copy()
        for j,v in enumerate(design_vars):
            row[v] = float(Xvar[k, j])
        dfk = add_quick_features(pd.DataFrame([row]))
        pred = float(predict_sse_from_matrix(dfk[all_num_cols].values, model, transform_kind, use_smear, smear_factor)[0])
        keep_rows.append({**dfk.iloc[0][['filename'] + design_vars].to_dict(), "pred_SSE": pred})
    cand_df = pd.DataFrame(keep_rows).sort_values("pred_SSE", ascending=False).reset_index(drop=True)
    log_df = pd.DataFrame(records)
    return cand_df, log_df

# ---- 실행: 앵커 선택 → 최적화 → 저장
# 앵커는 OOF 기준 상위 구조를 추천(없으면 실제 y 상위)
try:
    oof_path = os.path.join(SAVE_DIR, "oof_predictions_best.csv")  # 기존 산출물 경로
    oof = pd.read_csv(oof_path)
    anchors = oof.sort_values("y_pred_oof", ascending=False).head(5)['filename'].tolist()
except Exception:
    anchors = df.sort_values(target, ascending=False).head(5)['filename'].tolist()

ALL_RESULTS = []
for fname in anchors:
    anchor_row = df[df['filename']==fname].iloc[0]
    cand_df, log_df = cem_optimize_anchor(
        anchor_row=anchor_row,
        model=final_model_all,
        design_vars=DESIGN_VARS,
        bounds=bounds,
        all_num_cols=ALL_NUM_COLS,
        transform_kind=TRANSFORM_KIND,
        use_smear=(TRANSFORM_KIND=='log1p' and True),
        smear_factor=1.0  # CV 평균 스미어링 계수 있으면 넣어도 됨
    )
    out_dir = os.path.join(SAVE_DIR_INV, f"anchor_{fname}")
    os.makedirs(out_dir, exist_ok=True)
    cand_df.to_csv(os.path.join(out_dir, "candidates_feature_space.csv"), index=False)
    log_df.to_csv(os.path.join(out_dir, "cem_trace.csv"), index=False)
    print(f"[SAVE] {out_dir}/candidates_feature_space.csv")
    ALL_RESULTS.append(cand_df.assign(anchor=fname))

all_cands = pd.concat(ALL_RESULTS, ignore_index=True)
all_cands.sort_values("pred_SSE", ascending=False).to_csv(
    os.path.join(SAVE_DIR_INV, "all_candidates_feature_space.csv"), index=False
)
print(f"[DONE] inverse design candidates -> {os.path.join(SAVE_DIR_INV, 'all_candidates_feature_space.csv')}")


In [ ]:
# ---------------------------
# 12) SHAP 해석 (최종 모델 기준)
# ---------------------------
N_SHAP = min(5000, X_all.shape[0])
rng_shap = np.random.default_rng(SEED_BASE)
idx_shap = rng_shap.choice(np.arange(X_all.shape[0]), size=N_SHAP, replace=False)
X_shap_df = pd.DataFrame(X_all[idx_shap], columns=num_cols)

explainer = shap.TreeExplainer(final_model_all)
shap_values = explainer.shap_values(X_shap_df)

# SHAP summary (dot)
shap.summary_plot(shap_values, X_shap_df, show=False, plot_type="dot", cmap = 'jet', max_display = 20)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_dot.png"), dpi=160)
plt.close()

# SHAP summary (bar)
shap.summary_plot(shap_values, X_shap_df, show=False, plot_type="bar", max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "shap_summary_bar.png"), dpi=160)
plt.close()

print("[SAVE] SHAP summaries saved.")

# ---------------------------
# 13) 요약/모델 저장
# ---------------------------
hp_summary = {
    "config": {
        "use_quick_features": bool(USE_QUICK_FEATURES),
        "transform": TRANSFORM_KIND,
        "use_smearing": bool(USE_SMEARING and smearable),
        "outer_splits": int(min(N_SPLITS, n_groups)),
        "inner_splits": int(INNER_SPLITS),
        "seed_base": int(SEED_BASE),
        "n_features": int(len(num_cols)),
        "n_samples": int(len(df)),
        "n_groups": int(n_groups),
        "hp_trials": int(N_HP_TRIALS),
        "hp_runs_per_trial": int(N_HP_RUNS),
        "oof_runs_best": int(N_OOF_RUNS_BEST)
    },
    "best_params": {
        "learning_rate": float(best_params["learning_rate"]),
        "max_depth": int(best_params["max_depth"]),
        "min_child_weight": int(best_params["min_child_weight"]),
        "subsample": float(best_params["subsample"]),
        "colsample_bytree": float(best_params["colsample_bytree"]),
        "reg_lambda": float(best_params["reg_lambda"]),
        "reg_alpha": float(best_params["reg_alpha"])
    },
    "search_best": {
        "r2_mean": float(best_row['r2_mean']),
        "r2_std":  float(best_row['r2_std']),
        "mse_mean": float(best_row['mse_mean']),
        "mse_std":  float(best_row['mse_std']),
        "best_iter_median": int(best_iter_from_search),
        "smearing_mean": float(c_cv_from_search)
    },
    "oof_stats_best": oof_stats,
    "final_model": {
        "n_estimators": int(best_iter_final),
        "smearing_factor_cv_mean": float(c_cv_final)
    }
}
with open(os.path.join(SAVE_DIR, "summary.json"), "w") as f:
    json.dump(hp_summary, f, indent=2)

final_model_all.save_model(os.path.join(SAVE_DIR, "final_model_all.json"))
print(f"[SAVE] outputs -> {SAVE_DIR}")

In [ ]:
df.columns

In [ ]:
shap_df

In [ ]:
df.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (질문에서 사용한 설정 그대로)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

def _find_y_col(df: pd.DataFrame, preferred='sse') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용, 후보 자동 탐색)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def _categorize_x(series: pd.Series, bins=5):
    """
    (기본 동작) x축이 수치형이면 분위수(q=bins)로 구간화, 범주형/문자면 그대로 카테고리 반환.
    ※ 정수 카테고리로 강제하려면 boxplot_by_x()의 x_discrete_values를 쓰세요.
    """
    s = series.copy()
    s_num = pd.to_numeric(s, errors='coerce')
    if s_num.notna().mean() >= 0.9:
        q = min(bins, max(2, s_num.nunique()))
        try:
            return pd.qcut(s_num, q=q, duplicates='drop')
        except ValueError:
            return pd.cut(s_num, bins=q)
    else:
        return s.astype('category')

def boxplot_by_x(
    df: pd.DataFrame,
    x_col: str,
    y_col: str = None,
    bins: int = 5,
    title: str = None,
    show_means: bool = True,
    x_discrete_values: list | None = None,  # ← 여기에 [1,2,3,4,5] 같이 정수 카테고리를 넘기면 그대로 사용합니다.
):
    """
    y축: maximum splitting energy (또는 y_col 지정)
    x축: x_col (범주형이면 그대로, 수치형이면 기본은 분위수 binning)
         단, x_discrete_values가 주어지면 해당 값들을 '정수 카테고리'로 사용
    """
    if y_col is None:
        y_col = _find_y_col(df, preferred='sse')

    # 필요한 열만 가져오고 수치/결측 처리
    data = df[[x_col, y_col]].copy()
    data[y_col] = pd.to_numeric(data[y_col], errors='coerce')
    data = data.dropna(subset=[x_col, y_col])

    if x_discrete_values is not None:
        # 정수 카테고리 강제: 숫자로 변환 → 반올림/정수화 → 지정 카테고리 순서로 정렬
        xnum = pd.to_numeric(data[x_col], errors='coerce')
        mask = xnum.notna()
        data = data.loc[mask].copy()
        xint = np.rint(xnum.loc[mask]).astype(int)
        data['__xgroup'] = pd.Categorical(xint, categories=x_discrete_values, ordered=True)
    else:
        # 기본: 자동 카테고리화/분위수 구간화
        data['__xgroup'] = _categorize_x(data[x_col], bins=bins)

    # 그룹별 y 수집 (카테고리 순서 유지)
    groups, labels = [], []
    # observed=True면 비어 있는 카테고리는 생략됩니다(데이터가 없는 1~5 구간이 있다면 표시되지 않음)
    for grp, sub in data.groupby('__xgroup', observed=True):
        yvals = sub[y_col].values
        if len(yvals) == 0:
            continue
        groups.append(yvals)
        # 정수 카테고리면 그대로 라벨, 아니면 문자열화
        labels.append(str(grp))

    # 박스플롯
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, labels=labels, showmeans=show_means)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('N', fontsize=15)
    plt.xticks(rotation=0, fontsize=15)
    plt.yticks(fontsize=15)
    plt.tight_layout()
    plt.show()

# ===== 사용 예시 =====
# p_orb_e_non이 자연수 1~5라면 다음처럼 호출하세요.
boxplot_by_x(
    df,
    x_col='min_match_rmsd',
    y_col='sse',
    #x_discrete_values=[0, 1, 2, 3, 4, 5],   # ← 이 줄이 핵심: x축을 1,2,3,4,5로 고정
    #title='p_orb_e_non vs maximum splitting energy'
)

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df['p_d_elec_ratio'], df['sse'])
plt.xlabel('p_d_elec_ratio', fontsize = 15)
plt.ylabel('SSE (eV)', fontsize = 15)
#plt.tick_params(fontsize = 15)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Setting font configurations
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

imp = ['min_match_rmsd']
X = df.drop(columns=['filename', target]).select_dtypes(include=[np.number])
# 1) shap_vals → DataFrame 으로 변환 (각 열 이름은 X.columns, 인덱스는 df_full.index)
shap_df = pd.DataFrame(
    mean_shap_values_sse, 
    columns=X.columns, 
    index=df.index
)
import matplotlib.pyplot as plt

# 미리 shap_df, df_full, imp 리스트, 그리고 'is_octa' 플래그가 준비되어 있다고 가정

for col in imp:
    shap_col = shap_df[col]            # 해당 피처의 SHAP 값 (Series)

    # 음수/양수 SHAP 마스크
    neg_mask = (shap_col <  -0.) 
    pos_mask = (shap_col >= 0.)

    cmap = plt.get_cmap('bwr')

    plt.figure(figsize=(7,4))
    # — 음수 SHAP: 동그라미
    plt.scatter(
        df.loc[neg_mask, col],
        df.loc[neg_mask, 'sse'],
        c=shap_col[neg_mask],
        cmap=cmap,
        vmin=-0.1, vmax=0.1,       # 컬러바 범위 고정
        marker='o',
        s=30,
        edgecolor='k',
        alpha=0.8,
        label='SHAP < 0'
    )
    # — 양수 SHAP: 삼각형
    #plt.ylabel('Maximum Spin\nSplitting Energy (eV)', fontsize = 20)
    #plt.xticks(fontsize = 20)
    #plt.yticks(fontsize = 20)
    #plt.title(f"{col}: Octahedron samples colored by SHAP value")
    #plt.legend(frameon=True, fontsize = 13)
    #cbar = plt.colorbar()
    #cbar.ax.tick_params(labelsize=20)
    #cbar.set_label("SHAP value", size = 20)
    #cbar.set_ticks([-0.2, -0.1, 0, 0.1, 0.2])
    #plt.tight_layout()
    #plt.show()

    #plt.figure(figsize=(7,4))
    plt.scatter(
        df.loc[pos_mask, col],
        df.loc[pos_mask, 'sse'],
        c=shap_col[pos_mask],
        cmap=cmap,
        vmin=-0.1, vmax=0.1,       # 컬러바 범위 고정
        marker='^',
        s=30,
        edgecolor='k',
        alpha=0.,
        label='SHAP ≥ 0'
    )

    #plt.xlabel(r"$\angle$XMX standard deviation", fontsize = 20)
    #plt.xlabel(r"Motif - Motif distance (Å)", fontsize = 20)
    #plt.xlabel(r"X atom electronegativity", fontsize = 20)
    #plt.xlabel(r"Number of p electrons of the X atom", fontsize = 20)
    #plt.xlabel(r"Rotational angle of motif", fontsize = 20)
    #plt.xlim(0,0.4)
    plt.ylabel('Maximum Spin\nSplitting Energy (eV)', fontsize = 20)
    plt.xticks(fontsize = 20)
    plt.yticks(fontsize = 20)
    #plt.title(f"{col}: Octahedron samples colored by SHAP value")
    #plt.legend(frameon=True, fontsize = 13)
    cbar = plt.colorbar()
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label("SHAP value", size = 20)
    #cbar.set_ticks([-0.2, -0.1, 0, 0.1, 0.2])
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 0) 글꼴 설정
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

target_col = 'p_orb_e_non'  # y축 대상

for col in imp:
    shap_col = shap_df[col]

    # 1) 부호 마스크
    neg_mask = shap_col < 0
    pos_mask = shap_col >= 0

    # 2) 각 그룹의 타깃(series) 수치화 & 결측 제거
    y_neg = pd.to_numeric(df.loc[neg_mask, target_col], errors='coerce').dropna().values
    y_pos = pd.to_numeric(df.loc[pos_mask, target_col], errors='coerce').dropna().values

    labels = [f'{col} SHAP < 0 (n={len(y_neg)})', f'{col} SHAP ≥ 0 (n={len(y_pos)})']
    data = [y_neg, y_pos]

    # 3) 박스플롯
    plt.figure(figsize=(7, 4))
    plt.boxplot(data, labels=labels, showmeans=True)
    #plt.ylabel('Maximum Spin\nSplitting Energy (eV)', fontsize=20)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (원하시면 유지)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

def _find_y_col(df: pd.DataFrame, preferred='maximum splitting energy') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def boxplot_by_x_log_bins(
    df: pd.DataFrame,
    x_col: str,                    # 예: 'pd_ratio'
    y_col: str = None,             # 예: 'maximum splitting energy'
    n_bins: int = 10,              # 로그 기준 등간격 구간 개수
    x_min: float | None = None,    # None이면 데이터의 양의 최소값 사용
    x_max: float | None = None,    # None이면 데이터의 최대값 사용
    title: str | None = None,
    show_means: bool = False,
    label_decimals: int = 1        # x축 라벨 소수점 자리수
):
    # y열 결정 및 수치/결측 처리
    if y_col is None:
        y_col = _find_y_col(df, preferred='maximum splitting energy')
    data = df[[x_col, y_col]].copy()
    data[x_col] = pd.to_numeric(data[x_col], errors='coerce')
    data[y_col] = pd.to_numeric(data[y_col], errors='coerce')
    data = data.dropna(subset=[x_col, y_col])

    # 로그 스케일 유효 범위(양수)로 필터
    pos_mask = data[x_col] > 0
    data = data.loc[pos_mask].copy()
    if data.empty:
        raise ValueError(f"'{x_col}'에 양수 값이 없습니다. 로그 스케일을 사용할 수 없습니다.")

    # 범위 자동/지정
    x_min_data = data[x_col].min()
    x_max_data = data[x_col].max()
    xmin = x_min if (x_min is not None and x_min > 0) else x_min_data
    xmax = x_max if (x_max is not None and x_max > 0) else x_max_data
    if xmin <= 0 or xmax <= 0 or xmin >= xmax:
        raise ValueError(f"유효하지 않은 범위: x_min={xmin}, x_max={xmax}")

    # 로그 등간격 경계/구간
    edges = np.logspace(np.log10(xmin), np.log10(xmax), n_bins + 1)
    data['__xgroup'] = pd.cut(
        data[x_col], bins=edges, include_lowest=True, right=True
    )

    # 각 구간의 y값, 위치(기하중심), 상자 너비 수집
    groups, positions, widths, tick_labels = [], [], [], []
    for inter, sub in data.groupby('__xgroup', observed=True):
        yvals = sub[y_col].values
        if len(yvals) == 0:
            continue
        groups.append(yvals)

        # 기하중심(로그 중심): sqrt(left * right)
        center = np.sqrt(inter.left * inter.right)
        positions.append(center)

        # 상자 너비를 구간 폭의 일부로(로그-스케일에서도 자연스럽게 보이도록)
        widths.append((inter.right - inter.left) * 0.8)

        # 라벨(구간 중심, 소수점 1자리)
        tick_labels.append(f"{center:.{label_decimals}f}")

    # 박스플롯 (로그 x축에서 실제 좌표에 배치)
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, positions=positions, widths=widths, showmeans=show_means)
    #plt.xscale('log')
    plt.xlim(edges[0], edges[-1])
    plt.xticks(None)

    # x축 눈금/라벨을 구간 중심으로
    plt.xticks(positions, tick_labels, rotation=0, fontsize=15)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('p/d Ratio', fontsize=15)
    plt.yticks(fontsize=15)
    plt.tight_layout()
    plt.show()

# ===== 사용 예시 =====
# pd_ratio를 로그축으로 10개 구간(log-spaced) 나눠서 그리기
boxplot_by_x_log_bins(
    df,
    x_col='pd_ratio',
    y_col='maximum splitting energy',
    n_bins=10,
    # 필요 시 범위 지정(예: 0.01 ~ 2.5). 지정하지 않으면 데이터의 (양수) 최소/최대를 사용합니다.
    # x_min=0.01, x_max=2.5,
    #title='p/d Ratio (log-spaced) vs Maximum splitting energy',
    show_means=False,
    label_decimals=1   # x축 라벨을 소수점 첫째 자리로
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (질문에서 사용한 설정 그대로)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

def _find_y_col(df: pd.DataFrame, preferred='maximum splitting energy') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용, 후보 자동 탐색)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def boxplot_by_x_equal_bins(
    df: pd.DataFrame,
    x_col: str,                       # 예: 'pd_ratio'
    y_col: str = None,                # 예: 'maximum splitting energy'
    x_min: float = 0.0,
    x_max: float = 2.5,
    n_bins: int = 10,
    title: str = None,
    show_means: bool = False
):
    """
    y축: maximum splitting energy (또는 y_col 지정)
    x축: x_col을 [x_min, x_max] 범위에서 등간격 n_bins로 구간화하여 박스플롯 작성
    x축 라벨: 각 구간의 중심값을 소수점 첫째 자리로 표기
    """
    if y_col is None:
        y_col = _find_y_col(df, preferred='maximum splitting energy')

    # 필요한 열만 추출 및 수치/결측 처리
    data = df[[x_col, y_col]].copy()
    data[x_col] = pd.to_numeric(data[x_col], errors='coerce')
    data[y_col] = pd.to_numeric(data[y_col], errors='coerce')
    data = data.dropna(subset=[x_col, y_col])

    # 지정 범위로 필터
    mask = (data[x_col] >= x_min) & (data[x_col] <= x_max)
    data = data.loc[mask].copy()

    # 등간격 구간 생성
    edges = np.linspace(x_min, x_max, n_bins + 1)  # n_bins개의 구간 => n_bins+1개의 경계
    data['__xgroup'] = pd.cut(
        data[x_col],
        bins=edges,
        include_lowest=True,    # 0 포함
        right=True              # 오른쪽 폐구간: 마지막 구간이 (.., 2.5]가 되어 2.5 포함
    )

    # 그룹별 y 수집 + 라벨(구간 중심값을 소수점 1자리로)
    groups, labels = [], []
    # groupby는 카테고리 순서를 유지
    for inter, sub in data.groupby('__xgroup', observed=True):
        yvals = sub[y_col].values
        if len(yvals) == 0:
            continue
        groups.append(yvals)
        # 구간 중심값 -> 소수점 첫째 자리
        center = (inter.left + inter.right) / 2
        labels.append(f"{center:.1f}")

    # 박스플롯
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, labels=labels, showmeans=show_means)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('p/d Ratio (bin center)', fontsize=15)
    plt.xticks(rotation=0, fontsize=15)  # 라벨은 0.1, 0.4, ... 식으로 1자리 소수
    plt.yticks(fontsize=15)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()

# 사용 예시
boxplot_by_x_equal_bins(
    df,
    x_col='pd_ratio',
    y_col='maximum splitting energy',
    x_min=0.0,
    x_max=2.5,
    n_bins=10,
    #title='p/d Ratio vs Maximum splitting energy'
)


In [ ]:
sc = plt.scatter(
    df['pd_ratio'],
    df['maximum splitting energy'],
    c=df['ion1 tot'],
    cmap='jet'
)
plt.xscale('log')

cbar = plt.colorbar(sc)
cbar.set_label('ion1_tot')
plt.tight_layout()

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (질문에서 사용한 설정 그대로)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

def _find_y_col(df: pd.DataFrame, preferred='maximum splitting energy') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용, 후보 자동 탐색)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def _categorize_x(series: pd.Series, bins=5):
    """
    (기본 동작) x축이 수치형이면 분위수(q=bins)로 구간화, 범주형/문자면 그대로 카테고리 반환.
    ※ 정수 카테고리로 강제하려면 boxplot_by_x()의 x_discrete_values를 쓰세요.
    """
    s = series.copy()
    s_num = pd.to_numeric(s, errors='coerce')
    if s_num.notna().mean() >= 0.9:
        q = min(bins, max(2, s_num.nunique()))
        try:
            return pd.qcut(s_num, q=q, duplicates='drop')
        except ValueError:
            return pd.cut(s_num, bins=q)
    else:
        return s.astype('category')

def boxplot_by_x(
    df: pd.DataFrame,
    x_col: str,
    y_col: str = None,
    bins: int = 5,
    title: str = None,
    show_means: bool = True,
    x_discrete_values: list | None = None,  # ← 여기에 [1,2,3,4,5] 같이 정수 카테고리를 넘기면 그대로 사용합니다.
):
    """
    y축: maximum splitting energy (또는 y_col 지정)
    x축: x_col (범주형이면 그대로, 수치형이면 기본은 분위수 binning)
         단, x_discrete_values가 주어지면 해당 값들을 '정수 카테고리'로 사용
    """
    if y_col is None:
        y_col = _find_y_col(df, preferred='maximum splitting energy')

    # 필요한 열만 가져오고 수치/결측 처리
    data = df[[x_col, y_col]].copy()
    data[y_col] = pd.to_numeric(data[y_col], errors='coerce')
    data = data.dropna(subset=[x_col, y_col])

    if x_discrete_values is not None:
        # 정수 카테고리 강제: 숫자로 변환 → 반올림/정수화 → 지정 카테고리 순서로 정렬
        xnum = pd.to_numeric(data[x_col], errors='coerce')
        mask = xnum.notna()
        data = data.loc[mask].copy()
        xint = np.rint(xnum.loc[mask]).astype(int)
        data['__xgroup'] = pd.Categorical(xint, categories=x_discrete_values, ordered=True)
    else:
        # 기본: 자동 카테고리화/분위수 구간화
        data['__xgroup'] = _categorize_x(data[x_col], bins=bins)

    # 그룹별 y 수집 (카테고리 순서 유지)
    groups, labels = [], []
    # observed=True면 비어 있는 카테고리는 생략됩니다(데이터가 없는 1~5 구간이 있다면 표시되지 않음)
    for grp, sub in data.groupby('__xgroup', observed=True):
        yvals = sub[y_col].values
        if len(yvals) == 0:
            continue
        groups.append(yvals)
        # 정수 카테고리면 그대로 라벨, 아니면 문자열화
        labels.append(str(grp))

    # 박스플롯
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, labels=labels, showmeans=show_means)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('Number of p electrons of X', fontsize=15)
    plt.xticks(rotation=0, fontsize=15)
    plt.yticks(fontsize=15)
    plt.tight_layout()
    plt.show()

# ===== 사용 예시 =====
# p_orb_e_non이 자연수 1~5라면 다음처럼 호출하세요.
boxplot_by_x(
    df,
    x_col='p_orb_e_non',
    y_col='maximum splitting energy',
    x_discrete_values=[1, 2, 3, 4, 5],   # ← 이 줄이 핵심: x축을 1,2,3,4,5로 고정
    title='p_orb_e_non vs maximum splitting energy'
)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (질문에서 사용한 설정 그대로)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

HALOGENS = ['F', 'Cl', 'Br', 'I']  # x축 표시 순서

def _find_y_col(df: pd.DataFrame, preferred='maximum splitting energy') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용, 후보 자동 탐색)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def _extract_composition_from_filename(s: str) -> str:
    """
    '.../POSCAR_<composition>[.ext]' 또는 'POSCAR-<composition>' 형태에서 <composition>만 반환.
    형식이 다르면 파일명(확장자 제거) 전체를 반환.
    """
    base = os.path.basename(str(s))
    name, _ = os.path.splitext(base)
    m = re.search(r'POSCAR[_-](.+)$', name, flags=re.I)
    return m.group(1) if m else name

def _pick_single_halogen(comp: str) -> str | None:
    """
    조성 문자열 comp에서 원소 토큰을 분해해 F/Cl/Br/I가 '정확히 하나' 있는 경우 그 심볼을 반환.
    (여러 개 포함되면 None 반환하여 해당 샘플 제외)
    """
    tokens = re.findall(r'[A-Z][a-z]?', str(comp))
    present = [t for t in tokens if t in HALOGENS]
    if len(present) == 1:
        return present[0]
    return None  # 0개 또는 2개 이상이면 제외

def boxplot_halogen_from_filename(
    df: pd.DataFrame,
    y_col: str | None = None,
    show_means: bool = True,
    title: str | None = "Halogen vs Maximum splitting energy"
):
    # y 컬럼 결정 및 수치화
    if y_col is None:
        y_col = _find_y_col(df, preferred='maximum splitting energy')
    work = df[['filename', y_col]].copy()
    work[y_col] = pd.to_numeric(work[y_col], errors='coerce')
    work = work.dropna(subset=['filename', y_col])

    # filename -> composition -> 단일 할로겐 추출
    work['__comp'] = work['filename'].map(_extract_composition_from_filename)
    work['__halogen'] = work['__comp'].map(_pick_single_halogen)
    work = work.dropna(subset=['__halogen'])

    # 카테고리 순서를 F, Cl, Br, I로 고정
    work['__halogen'] = pd.Categorical(work['__halogen'], categories=HALOGENS, ordered=True)

    # 각 할로겐 그룹의 y값 수집 (비어있는 그룹은 자동 생략)
    groups, labels = [], []
    for h in HALOGENS:
        vals = work.loc[work['__halogen'] == h, y_col].values
        if len(vals) > 0:
            groups.append(vals)
            labels.append(h)

    if not groups:
        raise ValueError("F/Cl/Br/I가 포함된 샘플을 filename에서 찾지 못했습니다. filename 형식을 확인해 주세요.")

    # 박스플롯
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, labels=labels, showmeans=show_means)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('Halogen (from filename)', fontsize=15)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()

# === 사용 예시 ===
boxplot_halogen_from_filename(
    df,
    y_col='maximum splitting energy',  # 생략 가능(자동 탐지)
    show_means=True,
    title='Halogen (F/Cl/Br/I) vs Maximum splitting energy'
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 글꼴 설정 (질문에서 사용한 설정 그대로)
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

def _find_y_col(df: pd.DataFrame, preferred='maximum splitting energy') -> str:
    """y축에 쓸 'maximum splitting energy' 열을 찾습니다(대소문자/공백 차이 허용, 후보 자동 탐색)."""
    for c in df.columns:
        if str(c).strip().lower() == preferred.lower():
            return c
    lower = {c: str(c).lower() for c in df.columns}
    cand = [c for c,l in lower.items() if ('split' in l and 'energy' in l)]
    if not cand:
        cand = [c for c,l in lower.items() if ('max' in l and 'split' in l)]
    if cand:
        return cand[0]
    raise KeyError(f"'{preferred}'(y축) 열을 찾지 못했습니다. 사용 가능한 열: {list(df.columns)}")

def _categorize_x(series: pd.Series, bins=5):
    """
    (기본 동작) x축이 수치형이면 분위수(q=bins)로 구간화, 범주형/문자면 그대로 카테고리 반환.
    ※ 정수 카테고리로 강제하려면 boxplot_by_x()의 x_discrete_values를 쓰세요.
    """
    s = series.copy()
    s_num = pd.to_numeric(s, errors='coerce')
    if s_num.notna().mean() >= 0.9:
        q = min(bins, max(2, s_num.nunique()))
        try:
            return pd.qcut(s_num, q=q, duplicates='drop')
        except ValueError:
            return pd.cut(s_num, bins=q)
    else:
        return s.astype('category')

def boxplot_by_x(
    df: pd.DataFrame,
    x_col: str,
    y_col: str = None,
    bins: int = 5,
    title: str = None,
    show_means: bool = True,
    x_discrete_values: list | None = None,  # ← 여기에 [1,2,3,4,5] 같이 정수 카테고리를 넘기면 그대로 사용합니다.
):
    """
    y축: maximum splitting energy (또는 y_col 지정)
    x축: x_col (범주형이면 그대로, 수치형이면 기본은 분위수 binning)
         단, x_discrete_values가 주어지면 해당 값들을 '정수 카테고리'로 사용
    """
    if y_col is None:
        y_col = _find_y_col(df, preferred='maximum splitting energy')

    # 필요한 열만 가져오고 수치/결측 처리
    data = df[[x_col, y_col]].copy()
    data[y_col] = pd.to_numeric(data[y_col], errors='coerce')
    data = data.dropna(subset=[x_col, y_col])

    if x_discrete_values is not None:
        # 정수 카테고리 강제: 숫자로 변환 → 반올림/정수화 → 지정 카테고리 순서로 정렬
        xnum = pd.to_numeric(data[x_col], errors='coerce')
        mask = xnum.notna()
        data = data.loc[mask].copy()
        xint = np.rint(xnum.loc[mask]).astype(int)
        data['__xgroup'] = pd.Categorical(xint, categories=x_discrete_values, ordered=True)
    else:
        # 기본: 자동 카테고리화/분위수 구간화
        data['__xgroup'] = _categorize_x(data[x_col], bins=bins)

    # 그룹별 y 수집 (카테고리 순서 유지)
    groups, labels = [], []
    # observed=True면 비어 있는 카테고리는 생략됩니다(데이터가 없는 1~5 구간이 있다면 표시되지 않음)
    for grp, sub in data.groupby('__xgroup', observed=True):
        yvals = sub[y_col].values
        if len(yvals) == 0:
            continue
        groups.append(yvals)
        # 정수 카테고리면 그대로 라벨, 아니면 문자열화
        labels.append(str(grp))

    # 박스플롯
    plt.figure(figsize=(6, 4))
    plt.boxplot(groups, labels=labels, showmeans=show_means)
    plt.ylabel('Maximum spin\nsplitting energy (eV)', fontsize=15)
    plt.xlabel('Number of p electrons of X', fontsize=15)
    plt.xticks(rotation=0, fontsize=15)
    plt.yticks(fontsize=15)
    plt.tight_layout()
    plt.show()

# ===== 사용 예시 =====
# p_orb_e_non이 자연수 1~5라면 다음처럼 호출하세요.
boxplot_by_x(
    df,
    x_col='p_orb_e_non',
    y_col='ion1 tot',
    x_discrete_values=[1, 2, 3, 4, 5],   # ← 이 줄이 핵심: x축을 1,2,3,4,5로 고정
    title='p_orb_e_non vs maximum splitting energy'
)